# Taleemabad Coaching Framework — LLM Scoring **Wobble** Evaluation

Scores one classroom-observation transcript against **Sections B, C, D and F** of the
Coaching Framework using an open-weights instruct model that fits a **free Colab T4**,
repeats the whole evaluation *N* times, and then measures how much the scores **wobble**
run-to-run — per indicator, per section and overall — with confidence intervals,
reliability coefficients, significance tests and charts.

| | |
|---|---|
| **Framework** | 37 indicators — B: Lesson Plan Fidelity (10) · C: High-Leverage Practices (12) · D: Student Engagement (7) · F: Teacher Subject Knowledge (8) |
| **Scale** | whole numbers **1–4** (1 = Not Observed/Emerging → 4 = Highly Effective) |
| **Transcript** | session `8938cc17-0fe4-4323-a4f3-4af3d70d5fa5` — Urdu/English Grade-2 English reading lesson, 27 min (embedded; upload your own in §4) |
| **Model** | 4-bit NF4 Llama-3.1-8B-Instruct by default; Gemma-2-9B / Gemma-2-2B / Qwen2.5-7B selectable — all T4-safe |
| **Output** | tidy score table + per-indicator wobble table + section table + 7 charts + CSVs |

### What "wobble" means here
The transcript, the rubric and the prompt are **identical** on every run. The only thing
that changes is the model's sampling draw. So any score change between runs is pure
measurement noise. We quantify it four ways:

1. **Magnitude** — SD of the score across runs, and a bootstrap 95% CI on the mean.
2. **Agreement** — % of runs that hit the modal score; normalised entropy of the score distribution.
3. **Decision impact** — how often the run flips across the ≥3 "Proficient" line (the number that actually changes a coaching conversation).
4. **Significance** — per-indicator test that wobble exceeds a negligible rate (Holm-corrected), Krippendorff's α / ICC(2,1) / Fleiss' κ for overall reliability, and a Friedman test for run-to-run drift.

### How to run
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
Everything tunable lives in **§2 Hyperparameters** — change `TEMPERATURE` there and re-run
from §8 to watch the wobble move.

> ⏱ **Timing.** On a T4 with `SCORING_MODE="per_section"` (4 model calls per iteration), a
> 4-bit 8B model re-reads the whole ~12k-token transcript on every call, so expect roughly
> **3–8 min per iteration** → `N_ITERATIONS=10` ≈ 30–80 min. The run loop prints a projected
> total after iteration 1, so you'll know inside the first few minutes. To pilot fast, set
> `MODEL_KEY="gemma2-2b"` and `N_ITERATIONS=4` first; `SCORING_MODE="per_indicator"` (37 calls)
> is several times slower again, but scores each indicator in isolation.

---
## 1 · Environment

In [1]:
#@title Install dependencies (~2 min, first run only)
import subprocess, sys, os

# Must be set before torch initialises CUDA: lets the caching allocator grow segments
# instead of hoarding fixed-size blocks, which is what turns a tight run into a hard OOM.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PKGS = [
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
    "scipy>=1.11",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "jinja2",                 # pandas .style needs it
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=False)

import torch, transformers
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU          : {p.name}  |  {p.total_memory/1e9:.1f} GB  |  compute {p.major}.{p.minor}")
    IS_TURING = p.major == 7          # T4 = Turing 7.5 -> no bfloat16, no flash-attn
    print("bfloat16 ok  :", torch.cuda.is_bf16_supported())
else:
    IS_TURING = False
    print("\n*** No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all. ***")

torch        : 2.11.0+cu128
transformers : 5.13.1
CUDA         : True
GPU          : Tesla T4  |  15.6 GB  |  compute 7.5
bfloat16 ok  : True


In [2]:
#@title Hugging Face token (needed for the gated Llama / Gemma repos)
# Llama-3.1 and Gemma-2 are gated: accept the licence on the model page once, then
# put your token in Colab Secrets (key icon, left sidebar) as `HF_TOKEN`.
# Qwen2.5-7B-Instruct is UNGATED - use it if you would rather not deal with tokens.
import os

HF_TOKEN = ""
try:                                                    # Colab Secrets
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF token (blank = skip, ungated models only): ").strip()

if HF_TOKEN:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token set.")
else:
    print("No token - gated repos will 401. Set MODEL_KEY='qwen2.5-7b' in the next cell.")

HF token (blank = skip, ungated models only): ··········
HF token set.


---
## 2 · Hyperparameters — the knobs

Everything that can change a score lives here. The three that move wobble most:

| Knob | Effect on wobble |
|---|---|
| `TEMPERATURE` | The main dial. `0.0` = greedy (near-zero wobble, but also near-zero calibration insight). `0.3–0.7` is the realistic production band. `>1.0` degrades into noise. |
| `N_ITERATIONS` | Doesn't change wobble, changes how precisely you can *measure* it. 10 is the floor for a usable CI; 20–30 tightens it a lot. |
| `SCORING_MODE` | `per_section` = 4 calls/iteration, indicators see each other (cheap, mildly correlated). `per_indicator` = 37 calls/iteration, each score independent (slow, cleaner). |

`PROMPT_VARIANT` and `INDICATOR_ORDER` are there to separate *sampling* wobble from
*prompt-sensitivity* wobble — a rubric that scores differently just because the
indicators were shuffled has a bigger problem than temperature.

In [3]:
#@title ⚙️ CONFIG — edit, then Run all from here
import os, json
from dataclasses import dataclass, field, asdict

MODEL_ZOO = {
    # key                   repo id                              ctx    notes
    "llama3.1-8b":  dict(repo="meta-llama/Llama-3.1-8B-Instruct", ctx=131072, quant="4bit",
                         attn="sdpa",  gated=True,
                         note="DEFAULT. Best T4 quality/context trade-off. ~6 GB in 4-bit."),
    "gemma2-9b":    dict(repo="google/gemma-2-9b-it",             ctx=8192,   quant="4bit",
                         attn="eager", gated=True,
                         note="Strong scorer but only 8k ctx -> digest strategy forced. "
                              "Gemma-2 needs eager attention (logit soft-capping); fp16 can overflow."),
    "gemma2-2b":    dict(repo="google/gemma-2-2b-it",             ctx=8192,   quant="none",
                         attn="eager", gated=True,
                         note="Fast fp16 baseline. Noticeably wobblier - useful as the "
                              "'small model' arm of the comparison."),
    "qwen2.5-7b":   dict(repo="Qwen/Qwen2.5-7B-Instruct",         ctx=32768,  quant="4bit",
                         attn="sdpa",  gated=False,
                         note="UNGATED (no HF token needed). Best Urdu handling of the four."),
}

@dataclass
class Config:
    # ---------- model ----------
    MODEL_KEY: str        = "llama3.1-8b"   # any key from MODEL_ZOO above
    QUANT_OVERRIDE: str   = ""              # "" = zoo default | "4bit" | "8bit" | "none"

    # ---------- generation / sampling ----------
    TEMPERATURE: float    = 0.30            # ★ the wobble dial. 0.0 => greedy
    TOP_P: float          = 0.90
    TOP_K: int            = 50              # 0 disables top-k
    REPETITION_PENALTY: float = 1.05
    MAX_NEW_TOKENS: int   = 1100            # per_section needs room for 12 JSON objects
    BASE_SEED: int        = 1234            # iteration i uses BASE_SEED + i

    # ---------- experiment design ----------
    N_ITERATIONS: int     = 10              # >=10 for a usable CI; 20-30 to tighten it
    SECTIONS: tuple       = ("B", "C", "D", "F")
    SCORING_MODE: str     = "per_section"   # "per_section" | "per_indicator"
    PROMPT_VARIANT: str   = "standard"      # "standard" | "terse" | "cot"
    INDICATOR_ORDER: str  = "fixed"         # "fixed" | "shuffled" (probes order sensitivity)
    INCLUDE_EVIDENCE: bool = True           # ask for a short evidence quote per score
    ALLOW_NA: bool        = True            # let the model return "NA" (see note in §7)
    MAX_RETRIES: int      = 2               # JSON re-asks before recording a parse failure

    # ---------- transcript handling ----------
    CONTEXT_STRATEGY: str = "auto"          # "auto" | "full" | "truncate" | "digest"
    MAX_CTX_TOKENS: int   = 10240           # ★ hard cap on the prompt window, regardless of
                                            # what the model advertises. Llama-3.1's 131k ctx
                                            # is unusable on a T4: attention prefill costs
                                            # O(L^2) memory, so a 17k-token prompt asks for
                                            # ~17 GB of scratch on a 14.5 GB card. Capping it
                                            # here is what forces the digest strategy in §6.
                                            # 0 = no cap (only safe on an A100/L4).
    CTX_HEADROOM: int     = 1600            # tokens reserved for prompt + completion
    DIGEST_CHUNK_TOKENS: int = 1500
    DIGEST_ONCE: bool     = True            # True = digest built once (isolates scoring wobble)

    # ---------- statistics ----------
    N_BOOTSTRAP: int      = 5000
    CI_LEVEL: float       = 0.95
    NEGLIGIBLE_DISAGREEMENT: float = 0.05   # H0 for the per-indicator wobble test
    MC_SIMS: int          = 20000           # Monte-Carlo draws for the vs-random test
    PROFICIENCY_CUT: int  = 3               # >=3 is "Proficient" - flips across this line matter
    ALPHA: float          = 0.05
    STATS_SEED: int       = 7

    # ---------- io ----------
    OUT_DIR: str          = "wobble_out"
    VERBOSE: bool         = True

CFG = Config()

# --- derived / validation -------------------------------------------------
assert CFG.MODEL_KEY in MODEL_ZOO, f"MODEL_KEY must be one of {list(MODEL_ZOO)}"
MODEL_SPEC = dict(MODEL_ZOO[CFG.MODEL_KEY])
if CFG.QUANT_OVERRIDE:
    MODEL_SPEC["quant"] = CFG.QUANT_OVERRIDE
if CFG.TEMPERATURE <= 0:
    print("NOTE: TEMPERATURE=0 -> greedy decoding. Wobble will be ~0 by construction "
          "(only kernel non-determinism remains). Use it as the control arm, not the answer.")

os.makedirs(CFG.OUT_DIR, exist_ok=True)

print(f"Model   : {MODEL_SPEC['repo']}  ({MODEL_SPEC['quant']}, ctx {MODEL_SPEC['ctx']:,})")
print(f"Sampling: T={CFG.TEMPERATURE}  top_p={CFG.TOP_P}  top_k={CFG.TOP_K}  rep={CFG.REPETITION_PENALTY}")
print(f"Design  : {CFG.N_ITERATIONS} iterations x sections {list(CFG.SECTIONS)}  mode={CFG.SCORING_MODE}")
print(f"Ctx cap : {CFG.MAX_CTX_TOKENS or 'none'} tokens "
      f"({'prompt window capped below the model maximum' if CFG.MAX_CTX_TOKENS else 'using the full model window'})")
print(f"Note    : {MODEL_SPEC['note']}")

Model   : meta-llama/Llama-3.1-8B-Instruct  (4bit, ctx 131,072)
Sampling: T=0.3  top_p=0.9  top_k=50  rep=1.05
Design  : 10 iterations x sections ['B', 'C', 'D', 'F']  mode=per_section
Note    : DEFAULT. Best T4 quality/context trade-off. ~6 GB in 4-bit.


---
## 3 · The Coaching Framework (embedded)

All four sections transcribed from the framework CSVs — 37 indicators, each carrying its four
level descriptors verbatim, the source-framework rationale, and (for Section C) the Training
Curriculum module levels. Edit any descriptor here and the change flows straight into the
scoring prompt.

Only the **level descriptors** go into the prompt. The training-module column is teacher-
development metadata, not scoring criteria, so it stays in `FRAMEWORK_DF` for the report
rather than in the model's context.

The `why` field is **deliberately excluded from the scoring prompt** (`INCLUDE_WHY_IN_PROMPT
= False`): it contains effect sizes and advocacy language that nudges the model upward.
It is kept for the report. Flip the flag if you want to measure that effect.

In [4]:
#@title FRAMEWORK — Section B · Lesson Plan Fidelity (10 indicators)
INCLUDE_WHY_IN_PROMPT = False     # see note above

SECTION_B = dict(
    code="B",
    title="Lesson Plan Fidelity",
    note=("Fidelity Score = (actions observed / actions prescribed) x 100%. "
          ">=85% High | 60-84% Medium | <60% Low. "
          "Judge whether the designed lesson was actually delivered."),
    indicators=[
        dict(code="B1", name="Instructional Clarity & Learning Objectives",
             l1="No clear learning objective stated. Activities lack purpose.",
             l2="Objective mentioned but vague or not referenced during lesson.",
             l3="Clear objective stated, referred to during lesson, linked to classroom activities.",
             l4="Objective co-constructed with students, revisited at close. Students can articulate what they are learning and why.",
             why="FICO V3 (B1) + TEACH (Lesson Facilitation) + HOTS Lesson Planning. Clear objectives lift task completion 15-20%."),
        dict(code="B2", name="Lesson Structure & Sequence",
             l1="No discernible structure; random activities.",
             l2="Some structure but missing key phases (intro/body/close).",
             l3="Clear I Do -> We Do -> You Do sequence. Logical flow with transitions.",
             l4="Logical flow with smooth transitions, recap, and closure activity. Students can follow the arc.",
             why="FICO V3 (B8) + TEACH + OECD. Structured lessons improve retention ~25% (Rosenshine)."),
        dict(code="B3", name="Activities & Tasks Alignment",
             l1="Activities unrelated to lesson objective.",
             l2="Some activities align but others are filler.",
             l3="Most activities directly support the learning objective.",
             l4="All activities purposefully scaffolded toward objective mastery. No wasted time.",
             why="FICO V3 (B3) + TEACH + HOTS. Objective-activity alignment is the strongest predictor of lesson effectiveness (Hattie d=0.56)."),
        dict(code="B4", name="Activation of Prior Knowledge",
             l1="No reference to what students already know.",
             l2="Brief mention but no student input sought.",
             l3="Teacher connects new content to previously taught material.",
             l4="Students actively recall and link prior knowledge; teacher builds on it.",
             why="FICO V3 (B4) + OECD (Cognitive Activation) + HOTS. Schema activation - Ausubel's meaningful learning."),
        dict(code="B5", name="Meaningful & Real-World Connections",
             l1="Content presented in isolation, no real-world link.",
             l2="Teacher mentions a connection but doesn't develop it.",
             l3="Content connected to students' lives or local context.",
             l4="Students generate their own connections; examples from their community.",
             why="FICO V3 (B5) + OECD + HOTS. Contextual relevance increases motivation and transfer."),
        dict(code="B6", name="Differentiation / Catering to Learning Levels",
             l1="One-size-fits-all delivery, no differentiation.",
             l2="Aware of different levels but no adapted tasks.",
             l3="Tasks differentiated for at least 2 ability groups.",
             l4="Multiple pathways offered; struggling students supported, advanced students stretched.",
             why="FICO V3 (B6) + TEACH + OECD + Inclusive Education. In multi-grade Pakistani classrooms differentiation is survival."),
        dict(code="B7", name="Use of Taleemabad Lesson Plan",
             l1="Taleemabad lesson plan not used at all.",
             l2="Plan open but teacher deviates significantly.",
             l3="Plan followed with minor contextual adaptations.",
             l4="Plan followed faithfully AND adapted intelligently to class needs.",
             why="FICO V3 core fidelity check. Without it, impact evaluation is meaningless."),
        dict(code="B8", name="Use of Prescribed Resources",
             l1="No Taleemabad resources (video, worksheet, manipulatives) used.",
             l2="Some resources used but not as intended.",
             l3="Key resources used as prescribed in lesson plan.",
             l4="All resources used effectively; teacher adds complementary materials.",
             why="FICO V3 (Bi) + TEACH. Resources are the delivery mechanism of the curriculum."),
        dict(code="B9", name="Time on Task / Time on Learning",
             l1="Less than 50% of class time spent on learning activities.",
             l2="50-69% on task (significant management/transition time lost).",
             l3="70-85% on task with efficient transitions.",
             l4="More than 85% on task; routines are automatic, transitions seamless.",
             why="TEACH (Time on Task) + OECD + HOTS. Every 10% increase in time on task = measurable learning gains."),
        dict(code="B10", name="Lesson Closure & Consolidation",
             l1="Lesson ends abruptly with no summary.",
             l2="Teacher rushes through a brief recap.",
             l3="Structured closure: recap key points, check understanding.",
             l4="Students summarize learning, connect to next lesson, self-assess.",
             why="FICO V3 (B8) + TEACH. Closure activates retrieval practice (Dunlosky et al., 2013)."),
    ])
print(f"Section B: {len(SECTION_B['indicators'])} indicators")

Section B: 10 indicators


In [5]:
#@title FRAMEWORK — Section C · High-Leverage Practices (12 indicators)
SECTION_C = dict(
    code="C",
    title="High-Leverage Practices (Teacher Pedagogy & Training Curriculum Alignment)",
    note="Judge the teacher's pedagogical moves. Evidence must be visible in the transcript.",
    indicators=[
        dict(code="C1", name="Quality Questioning (Bloom's Aligned)",
             l1="Only yes/no or recall questions asked. Close-ended, requiring one-word answers.",
             l2="Mix of recall and some open-ended questions, but they lack depth (e.g. 'Why is the capital important?' without further exploration).",
             l3="Purposeful mix including application & analysis questions. Open-ended questions dominate. Wait time given.",
             l4="Questions span all Bloom's levels (Remember->Create); students generate questions; Socratic questioning evident.",
             module="L0 5-step lesson plan | L1 Open-ended questions, Think-Pair-Share, Bloom's | L2 Socratic questioning",
             why="FICO V3 (C1) + TEACH + OECD + HOTS. Classrooms with higher-order questions show 2x learning gains."),
        dict(code="C2", name="Responsive Re-explanation & Adaptive Teaching",
             l1="Repeats same explanation when students don't understand.",
             l2="Tries a different approach but still teacher-centered.",
             l3="Uses alternative representations (visual, concrete, analogy). Adjusts teaching to student level.",
             l4="Diagnoses misconception, re-explains using student's own logic, confirms understanding.",
             module="L0 CPA approach | L1 Diagnosing misconceptions | L2 Comprehension strategies",
             why="FICO V3 (C2) + TEACH + OECD + HOTS scaffolding. Re-explanation separates trained from untrained teachers."),
        dict(code="C3", name="Effective Feedback",
             l1="No feedback given, or only 'good/bad' evaluations. Generic: 'Good job' or 'Try again.'",
             l2="Feedback given but generic ('try harder'). Specific but does not consistently guide improvement.",
             l3="Specific feedback on what was done well and what to improve. Actionable.",
             l4="Feedback is specific, actionable, with next steps. Students use feedback to self-correct. Guides refinement of reasoning.",
             module="L0 Positive verbal feedback, quick checks | L1 Formative assessment | L2 Rubrics, self & peer assessment",
             why="FICO V3 (C3) + TEACH + HOTS. Feedback d=0.73 - but only when specific and actionable."),
        dict(code="C4", name="Equitable Participation",
             l1="Only 2-3 students participate; others ignored. Teacher-dominated.",
             l2="Teacher calls on volunteers only. A few students contribute while others stay silent.",
             l3="Deliberate strategies: cold call, pair-share, name sticks. Diverse students included.",
             l4="All students participate; teacher tracks contributions; gender-equitable. Students debate and refine arguments.",
             module="L0 Inclusive education | L1 Student-centred strategies in overcrowded classrooms, peer teaching | L2 Differentiated instruction",
             why="FICO V3 (C4) + TEACH + OECD + HOTS. Participation skews male and front-row; HOTS requires ALL students."),
        dict(code="C5", name="Student Agency & Voice",
             l1="Students are passive recipients; no choice or voice. Content from single perspective.",
             l2="Occasional student input but teacher-dominated. Multiple perspectives mentioned but not explored.",
             l3="Students make choices about how to demonstrate learning. Explore multiple perspectives.",
             l4="Students lead discussions, choose methods, self-assess, peer-teach. Create novel solutions. Evaluate alternatives.",
             module="L1 Think-Pair-Share, brainstorming | L2 Student-led discussions, PBL | L3 Peer mentoring",
             why="FICO V3 (C5) + OECD + HOTS. Agency bridges compliance to ownership."),
        dict(code="C6", name="Classroom Management & Routines",
             l1="Frequent disruptions; no visible routines. Students struggle to engage.",
             l2="Some routines but inconsistently enforced. Instructions lack clarity for all groups.",
             l3="Clear routines (entry, transitions, dismissal); minimal disruptions. Expectations clear.",
             l4="Seamless routines; students self-manage; positive behavioural reinforcement. Students actively participate in complex, clearly defined tasks.",
             module="L0 Routines, rules, attention strategies | L1 SMART behaviour goals | L2 Restorative practices | L3 School-wide programs",
             why="TEACH Area 2 + OECD + HOTS. Classroom culture is prerequisite for all learning."),
        dict(code="C7", name="Positive & Supportive Learning Environment",
             l1="Negative tone; punitive language or humiliation.",
             l2="Neutral but cold; no encouragement.",
             l3="Warm, encouraging tone; mistakes treated as learning opportunities.",
             l4="Joyful classroom; students feel safe to take risks; laughter and curiosity present.",
             module="L0 Child psychology fundamentals | L1 UDL | L2 Restorative circles, emotional check-ins",
             why="TEACH + OECD + HOTS. Psychologically safe children learn 2x faster (Durlak et al., 2011)."),
        dict(code="C8", name="Modeling, Scaffolding & Problem-Solving",
             l1="Teacher tells but doesn't show. Simple tasks demonstrated without explanation of process.",
             l2="Teacher demonstrates once but moves on quickly. Problem-solving modeled but strategies not explained.",
             l3="I Do -> We Do -> You Do scaffolding visible. Problem-solving and creativity modeled with clear strategies.",
             l4="Gradual release with checks at each stage; scaffold removed when ready. Teacher brainstorms solutions and explains reasoning.",
             module="L0 5-step lesson plan | L1 Scaffolding, GRR model | L2 Inquiry-based learning, PBL",
             why="TEACH + OECD + HOTS. Vygotsky's ZPD in practice."),
        dict(code="C9", name="Collaborative Learning",
             l1="No group or pair work. Students work individually without interaction.",
             l2="Students in groups but working individually. Tasks lack depth.",
             l3="Purposeful pair/group tasks with clear roles. Students work towards synthesized solutions.",
             l4="Structured collaboration (think-pair-share, jigsaw); students build on each other's ideas. Teams design solutions to community problems.",
             module="L0 Group reading | L1 Peer teaching, TPS | L2 Small-group problem solving | L3 PBL showcases",
             why="OECD + HOTS. Collaboration must target synthesis and problem-solving, not just sitting together."),
        dict(code="C10", name="Integration of Taleemabad Technology",
             l1="No technology used despite availability.",
             l2="Technology used as distraction/babysitter.",
             l3="Taleemabad videos/apps used to support learning objectives.",
             l4="Technology integrated seamlessly; students interact with content; teacher facilitates around it.",
             module="L0 Digital literacy | L1 Blended learning, Google Forms | L2 Canva, Drive, Meet | L3 Flipped classroom",
             why="FICO V3 + Taleemabad curriculum design. Tech-enhanced teaching is the core value proposition."),
        dict(code="C11", name="Self & Peer Assessment Facilitation",
             l1="Assessment limited to teacher-led grading. Students receive grades without reflection.",
             l2="Some self- or peer-assessment occurs, but inconsistent. Students assess without clear criteria.",
             l3="Self- and peer-assessment structured and purposeful. Students use rubrics to assess work.",
             l4="Students use rubrics to assess work, suggest improvements for peers, and set goals. Assessment tasks require analysis/evaluation/creation.",
             module="L0 Checklists, verbal feedback | L1 Formative + summative | L2 Portfolios, peer review, rubric design | L3 Data dashboards",
             why="HOTS Assessment & Feedback + TEACH. Self/peer assessment builds metacognition."),
        dict(code="C12", name="Classroom Resources & Space for Collaboration",
             l1="Resources and space disorganized, limiting collaborative learning. No group work areas.",
             l2="Some organization, but space/resources do not fully support collaboration.",
             l3="Resources and space well-organized for collaborative tasks. Materials accessible.",
             l4="Tables arranged for group work, materials easily accessible. Environment designed for inquiry and collaboration.",
             module="L0 Visual/auditory aids | L1 Learning stations | L2 Assistive technologies | L3 Community projects",
             why="HOTS Classroom Environment + OECD. Space arrangement directly predicts collaboration quality."),
    ])
print(f"Section C: {len(SECTION_C['indicators'])} indicators")

Section C: 12 indicators


In [6]:
#@title FRAMEWORK — Section D · Student Engagement (7) + Section F · Subject Knowledge (8)
SECTION_D = dict(
    code="D",
    title="Student Engagement",
    note=("Observe STUDENT behaviours, not teacher actions. The framework asks for a sample of "
          "at least 5 students across different locations in the classroom - in a transcript, "
          "infer this from how many distinct students speak and how they respond."),
    indicators=[
        dict(code="D1", name="Active Participation Rate",
             l1="Less than 25% of students visibly engaged. Collaboration minimal or absent.",
             l2="25-50% engaged; many passive or off-task.",
             l3="50-75% actively participating (writing, discussing, solving).",
             l4="More than 75% actively engaged; energy is visible; students initiating. Structured collaboration on synthesis/problem-solving.",
             why="FICO V3 (D1) + TEACH + OECD + HOTS. The most direct measure of whether teaching is reaching students."),
        dict(code="D2", name="Cognitive Engagement Level (Bloom's)",
             l1="Students copying or doing rote recall only. Passively receiving information.",
             l2="Students completing tasks but without thinking deeply.",
             l3="Students applying concepts to new problems (Bloom's Apply/Analyze).",
             l4="Students creating, evaluating, debating - genuine intellectual work. Actively analyse, interpret and critique content with supporting evidence.",
             why="FICO V3 (D2) + HOTS + OECD. Being busy != being engaged."),
        dict(code="D3", name="Student-to-Student Interaction",
             l1="No peer interaction; silent individual work only.",
             l2="Students talk but not about content.",
             l3="Students discuss content in pairs/groups; academic language used.",
             l4="Students build on each other's ideas; respectful disagreement; peer teaching. Students debate solutions and propose creative alternatives.",
             why="FICO V3 (D3) + OECD + TEACH + HOTS. In 40+ student classrooms peer learning is a necessity."),
        dict(code="D4", name="Student Confidence & Risk-Taking",
             l1="Students afraid to answer; avoidance behaviours visible.",
             l2="Students answer only when certain; no risk-taking.",
             l3="Students attempt challenging tasks; some comfortable with mistakes.",
             l4="Students volunteer, ask questions, try difficult problems. Mistakes celebrated. Students freely share and debate ideas.",
             why="FICO V3 (D4) + OECD + TEACH + HOTS. Without risk-taking, higher-order thinking is impossible."),
        dict(code="D5", name="On-Task Behavior During Independent Work",
             l1="Most students off-task during independent/group work.",
             l2="Students start on-task but lose focus quickly.",
             l3="Students sustain focus for most of independent work time.",
             l4="Students self-regulate; seek help appropriately; persist through difficulty.",
             why="FICO V3 (D5) + TEACH + OECD. Independent work reveals whether teaching has transferred."),
        dict(code="D6", name="Student Use of Learning Materials",
             l1="Students don't interact with provided materials.",
             l2="Materials used passively (watching video, holding textbook).",
             l3="Students actively use materials to solve problems or practice.",
             l4="Students use materials creatively; extend beyond prescribed use.",
             why="TEACH + OECD. If students aren't actively using materials, the materials aren't working."),
        dict(code="D7", name="Inclusivity of Engagement",
             l1="Only front-row or high-ability students engaged.",
             l2="Teacher attempts inclusion but success is limited.",
             l3="Students across ability levels and genders are participating.",
             l4="Deliberate inclusion of marginalized students; no one invisible. Gender-equitable participation.",
             why="TEACH + OECD + FICO V3 (C4) + Inclusive Education. Gender and ability gaps start in the classroom."),
    ])

SECTION_F = dict(
    code="F",
    title="Teacher's Subject Knowledge",
    note=("Assessed through lesson observation: does the teacher demonstrate accurate, deep "
          "understanding of the content? F5/F6/F7 are subject-specific - only the one matching "
          "the lesson's subject applies; return NA for the others if ALLOW_NA is on."),
    indicators=[
        dict(code="F1", name="Content Accuracy",
             l1="Teacher makes factual errors that go uncorrected.",
             l2="Mostly accurate but with minor errors or imprecise language.",
             l3="Content is accurate; no errors observed.",
             l4="Content is accurate AND teacher explains WHY (conceptual depth, not just facts).",
             why="FICO V3 (B9) + TEACH + HOTS. 15-25% of teachers in LMICs make content errors."),
        dict(code="F2", name="Use of Academic Language",
             l1="Incorrect or no subject-specific terminology used.",
             l2="Some terms used but not explained or used inconsistently.",
             l3="Key terms used accurately and explained to students.",
             l4="Terms used naturally; students also use them; bilingual bridging (Urdu/English) effective.",
             why="FICO V3 (B10) + OECD + Content Expertise. Academic language is the medium of assessment."),
        dict(code="F3", name="Anticipation of Student Misconceptions",
             l1="Teacher unaware of common misconceptions in this topic.",
             l2="Aware but doesn't address them proactively.",
             l3="Anticipates and addresses at least 1-2 common misconceptions.",
             l4="Systematically surfaces and corrects misconceptions; uses diagnostic questions.",
             why="FICO V3 (B11) + TEACH + L1 diagnosing misconceptions. Shulman's PCK."),
        dict(code="F4", name="Depth of Explanation",
             l1="Superficial/procedural explanation only ('do it this way').",
             l2="Some conceptual explanation but relies on memorization.",
             l3="Explains the 'why' behind procedures; uses multiple representations.",
             l4="Deep conceptual teaching; connects to broader principles; encourages student reasoning.",
             why="HOTS + OECD + TEACH. A teacher who only teaches procedures produces students who only memorize."),
        dict(code="F5", name="Subject-Specific Pedagogy: MATH",
             l1="Math taught purely procedurally; no use of manipulatives or visuals.",
             l2="Some visual aids but conceptual understanding not developed.",
             l3="Uses concrete -> pictorial -> abstract (CPA) progression; manipulatives present.",
             l4="CPA approach mastered; multiple solution strategies explored; math talk norms established.",
             subject="MATH",
             why="FICO V3 (BM12-BM13) + EGMA + Content Expertise: Math. CPA is gold standard for primary math."),
        dict(code="F6", name="Subject-Specific Pedagogy: SCIENCE",
             l1="Science taught from textbook only; no inquiry or observation.",
             l2="Some demonstration but teacher-led; students observe passively.",
             l3="Hands-on activities present; students make predictions and observations.",
             l4="Full inquiry cycle: question -> predict -> investigate -> conclude. Students design investigations.",
             subject="SCIENCE",
             why="FICO V3 (BS12-BS13) + OECD + HOTS. Inquiry-based learning d=0.40."),
        dict(code="F7", name="Subject-Specific Pedagogy: LITERACY / LANGUAGE",
             l1="Reading taught as decoding only; no comprehension strategies.",
             l2="Some reading activities but no explicit strategy instruction.",
             l3="Teacher models reading strategies (prediction, summarizing, questioning). Balanced approach.",
             l4="Balanced literacy: phonics + fluency + vocabulary + comprehension + writing integrated.",
             subject="LITERACY",
             why="FICO V3 (BL12-BL14) + EGRA + Content Expertise: English & Urdu. Balanced literacy wins."),
        dict(code="F8", name="Cross-Curricular Connections",
             l1="Subject taught in complete isolation.",
             l2="Occasional reference to other subjects but not developed.",
             l3="Meaningful connections made to at least one other subject area.",
             l4="Integrated approach; students see how math connects to science connects to language.",
             why="OECD + HOTS + L2-L3 interdisciplinary PBL. Transfer is the ultimate goal of education."),
    ])

FRAMEWORK = {s["code"]: s for s in (SECTION_B, SECTION_C, SECTION_D, SECTION_F)}
print(f"Section D: {len(SECTION_D['indicators'])} indicators")
print(f"Section F: {len(SECTION_F['indicators'])} indicators")
print(f"TOTAL     : {sum(len(s['indicators']) for s in FRAMEWORK.values())} indicators "
      f"across {len(FRAMEWORK)} sections")

Section D: 7 indicators
Section F: 8 indicators
TOTAL     : 37 indicators across 4 sections


In [7]:
#@title Framework helpers — flat index + prompt rendering
import pandas as pd, textwrap

FRAMEWORK_DF = pd.DataFrame([
    dict(section=s["code"], section_title=s["title"], code=i["code"], indicator=i["name"],
         L1=i["l1"], L2=i["l2"], L3=i["l3"], L4=i["l4"],
         subject=i.get("subject", ""), module=i.get("module", ""), why=i.get("why", ""))
    for s in FRAMEWORK.values() for i in s["indicators"]
])
ALL_CODES   = FRAMEWORK_DF["code"].tolist()
CODE_ORDER  = {c: k for k, c in enumerate(ALL_CODES)}
CODE2NAME   = dict(zip(FRAMEWORK_DF["code"], FRAMEWORK_DF["indicator"]))
CODE2SECTION= dict(zip(FRAMEWORK_DF["code"], FRAMEWORK_DF["section"]))
SECTION_CODES = {s: FRAMEWORK_DF.loc[FRAMEWORK_DF.section == s, "code"].tolist()
                 for s in FRAMEWORK}

def render_indicator(ind, terse=False):
    """One indicator as a rubric block for the prompt."""
    if terse:
        return (f"{ind['code']} — {ind['name']}\n"
                f"  1={ind['l1']}\n  2={ind['l2']}\n  3={ind['l3']}\n  4={ind['l4']}")
    lines = [f"### {ind['code']} — {ind['name']}"]
    for lvl, key, label in ((1, "l1", "Not Observed / Emerging"), (2, "l2", "Developing"),
                            (3, "l3", "Proficient / Effective"), (4, "l4", "Highly Effective")):
        lines.append(f"  {lvl} ({label}): {ind[key]}")
    if ind.get("subject"):
        lines.append(f"  [Applies only to {ind['subject']} lessons]")
    if INCLUDE_WHY_IN_PROMPT and ind.get("why"):
        lines.append(f"  Context: {ind['why']}")
    return "\n".join(lines)

def render_section_rubric(section_code, codes=None, terse=False):
    s = FRAMEWORK[section_code]
    inds = [i for i in s["indicators"] if codes is None or i["code"] in codes]
    if codes is not None:                            # honour caller's order (shuffling)
        inds = sorted(inds, key=lambda i: codes.index(i["code"]))
    head = f"SECTION {s['code']} — {s['title'].upper()}\nSection guidance: {s['note']}"
    return head + "\n\n" + "\n\n".join(render_indicator(i, terse) for i in inds)

display(FRAMEWORK_DF[["section", "code", "indicator"]].head(8))
print(f"\n--- prompt preview -------------------------------------------------")
print(render_section_rubric("D")[:900] + " ...")

,section,code,indicator
0,B,B1,Instructional Clarity & Learning Objectives
1,B,B2,Lesson Structure & Sequence
2,B,B3,Activities & Tasks Alignment
3,B,B4,Activation of Prior Knowledge
4,B,B5,Meaningful & Real-World Connections
5,B,B6,Differentiation / Catering to Learning Levels
6,B,B7,Use of Taleemabad Lesson Plan
7,B,B8,Use of Prescribed Resources



--- prompt preview -------------------------------------------------
SECTION D — STUDENT ENGAGEMENT
Section guidance: Observe STUDENT behaviours, not teacher actions. The framework asks for a sample of at least 5 students across different locations in the classroom - in a transcript, infer this from how many distinct students speak and how they respond.

### D1 — Active Participation Rate
  1 (Not Observed / Emerging): Less than 25% of students visibly engaged. Collaboration minimal or absent.
  2 (Developing): 25-50% engaged; many passive or off-task.
  3 (Proficient / Effective): 50-75% actively participating (writing, discussing, solving).
  4 (Highly Effective): More than 75% actively engaged; energy is visible; students initiating. Structured collaboration on synthesis/problem-solving.

### D2 — Cognitive Engagement Level (Bloom's)
  1 (Not Observed / Emerging): Students copying or doing rote recall only. Passively receiving information.
  2 (Develop ...


---
## 4 · The transcript

The evaluated session is embedded below as a gzip+base64 blob so the notebook is
self-contained. To score a **different** session, run the upload cell and pick any JSON with
the same shape (`{"session_id": ..., "language": ..., "transcript": "[mm:ss] Speaker (LANG): ..."}`).

In [8]:
#@title Embedded session — 8938cc17 (Grade-2 English reading, Urdu medium, 27 min)
import base64, gzip, json

SESSION_B64 = (
    "H4sIADGPcmoC/8U9XW8bSXJ/ZeCnC2LpSM7wQ3q7AHtZIMAFyO7hcMgeDFocrYnViQYp2WcEAaKDRPCBLxH8bsBw"
    "TFnRhidLgSM/6HcMqZcgvyRdVf1R/TUz9N0mD2sNZ6qrq6urq6qrqnv/6dEkn0yGo8Mnw8Gj3eRRbyft7e01u1uN"
    "/TzbytJWutXP9tOtrL+fDrqNQXu/3370OHk0GR2P93JosTfq7z0bHn7/RCKa4Oejcf/o+Pf8O7w+nuRj2dFOv/u0"
    "97SXb/W6T5tbWTfd39rptQZbe43uXmtv0HzaTnNoctA//P64/z12dTyGN4OnTwbHAj0QPcn3RoeDCXxsdppd+Ly/"
    "/3w8eppHYbazDoD1jwfD0ZP94QFiNr8mT16kP1c0/7wOO7ZH3+Pg9sZ5/ygfPOkfAcZWo9XZanS3mr1vm43dVms3"
    "62yn7V6j0fjrRmO30YAWgkmHk73x8Dm2+Ed8/7vkm6PjQX54lPzsq1/91W7yt6Pt7w6/O8SPrd8l3+aCsnyc/OzX"
    "/yA+PpwUi/XnpLh6mBfzpFisTosb8d9idZYUd6vTh/n6w+rs4XUi/pkX18VCPKxmxUWxKN6vTpOHuWy/ul//q/hY"
    "XD+cPJwn66X4Pk9Wd+t7QJDAy+JNHLsmr8doR/JEXxrOam8aNRvumETnF9CzGNVqJkh7mK/ui+ukuBGkiV8nAqGg"
    "cbq+X92JQQH54t2seA8tLpHkz8iL4gaHLb6L7merqcCA4yreilGL4Z2spusPxS18eAuIsLWAeCPg6bf9WlPcdYcp"
    "+DYvPq5OBUFmXD1vXNcE446ktLNWw+sMRr00HbV8BiKEwMh4ImRAvsb+kIKz1RQEQcOIFqv7RJA4Wy/Ff0IspjDx"
    "uqPUI+VCcPJSsBpmA/rQoG2PJhdUdF98fDiJjttj8m/ziV4ILY+5kov2H0KqBk7CIKZXDPs9Mh7EXjwUF5IR58QI"
    "NuY0LSMjzVwyVicgC0KwZsVHoAIW0xqkV6ypIP5OKf6OvzaEpF+r8YkZPA3wLvXWoZAXkBk2Q+mOh1rCIFZLbqhP"
    "Eo6pllixhMTyuXmY4/ISAxQL6o6w+CRlpXzMMn86xeqtg9fjn2j5USxpaGtGm3UCHSgwG7cZsKEh1vmOp+/09DsE"
    "tANaDlYCslEogjPirPhnDjpPdI8SBAIKAiPlBlYNqg2Y+1PR2X9pSUBAI/IapaG27bNqJnoTU13cokYwxHY9sZ6B"
    "+YCpXgq6hGzcgpIW/90Kkmw0RucWlzAcYMQpqHDQR5cPMFpahtRbE42aTVeqSBEfU2/i7N5SIMDtSryPddYr68xb"
    "E2kUJUxFGVOkDIGFBfquheG7j7ImTm/Tm7SWobfZDUi11ZMQZ2nA5x75QCyYorlYXZsT1spKCPPVf6uUkVOgUtB9"
    "h8pZ/UC/IwHbj9Z1CtbV+vkFZKeNErLTVoCfhDmGrowLabtszYNgkIMDwnwjdBGtWVzIyhOT/UlvRlCzAOMF/hd+"
    "/u5RsWT27Q7durPvHgGfFFZspLSEaWvGkDV9a+EjNcPK0oi/geYOF8A1ehc4NzhONkRUYiR6ekjEBuroDtxSoG+u"
    "iHao7br+8d8NDwfJ13l/fDRJ+uLxm2f9sfDcBVR/PNmu+KzQtgNzpTluKAuSZCnWGiTpdjvBPo1QX6PjfAd+o3Jh"
    "tSdrhOON1UyueCG2M+Ir/iM5SkpJUd7abbRDUw8Cr6abYFxHgWCiWD3XbXUPrraF1XPgFIyWkUswpC4bmPYip+QS"
    "JAt3NtbcSMUmUApPWWlhudmB3QItsLmCVgBo5XVLjcteChFBl2TbsMCektUhfl+afZc3OEscpJ3H/dOdIs34uyd8"
    "UuSIcJ0xtUKYlo4jQR2g/4E+Hro8sO0Ctgq/Vfy7OtEeBnes0W3W894q8WVbQZdd7toEm67Qy5HKj1h1o51y3I++"
    "YRQ7tIlvoqfHyd8cT56N+9ulhlmRk7XcRfv16OBVMtpPjp7l9Px1//nzV8m3w9/nyXCCr4/g+eWz/DB5no+eH+TJ"
    "gVjjh7jE8UkP1nL3EPuv8j8cmc+p2/m3z/JJnvwwHEySZ/0XedJPDkZHRI1QHKL3UTIYPYY/2BE+PR/3946Ge7l5"
    "LTtIcQlu4MFpzoLwIDEC5ctDGPaLfPwqyf+wNzzKB9vJVzUetM20jWW624o7eiluLm2OgWcXp6R8PLYPsZt8NRwk"
    "xwdbvxwejQHV6FBMbz95mec/JP2X/Veab6lHImtpiPUtuwX2pb8U+iwrpSLSqOMykHGBNhSKGcDJ0WGuR23tY7Cp"
    "Bvlzngh9hiEth1sSBObi+fHREdjG4+fJ3uhgNN4/Pnisn5Lx8OnT0SFZ0af9g4OR+LGNfl5xvQKZnbMfihkZevLu"
    "tjDSpBpZsx2I5EDI4CPInXxk4IHwkAfTi6E0MIHIjw/TrIGnVQOPH8iICE9xAdsvcEKgvTZLtBnlflGGGt+LF7HG"
    "HM6WXKcTbzmDWojqym0ZqTAWT0ZHKQZKobc5hqFcVFpk005AQYfgPNJ/4RCjIb01FoPMmi7OcN++BYvAZXVpzDq1"
    "aex6NAoTNBr9YCC88boQfkjEl7mAvy0aNgNhF+FWyUZGsNppIDzHASusiAv80/wWfsyH4t/hg4rh0L53ikFhGXMD"
    "9wccf+03YrzWagEgfx5FxLf2bjOrxeB2SDPerKe4B4HgFQWzQKWKhhDAMg19HVmzYWjPRGEM2vpMrQ2PmN8TFccD"
    "CEgAvMPwHNsmqIYuSED42oEg+HpqyGv54aop+PMb99It7cW3HgsMhiHbWGfYtZgy/qCQ+K5OCMKdXQcircSRVeLI"
    "vhSHFBcG2Amj8gG7AQaeSy4yMN+AuYh2aiHKGlWIsmY9RM1KRK16iLIQohWxKhG68AIjaTNpfVds0WedkPSFmuoW"
    "fl5w6kNshrPdqMLZbm6Ms1WJMw1sYj0l1c5C3qcH1a6Fq1MLV7cWrm4tXL1auHbiuP77394RWCfg+3Nk7IdMtCww"
    "dSYszSnGTyj+g/mX9ecQIZ1A+iAIVWPiBFRWC1dtZzmcYBEYaqZiOoE4mZttOVnNjEnuhHY+5Q38/LvTwAognxWf"
    "xNRcWOZK4PAXjnKsMEd/A/+yLrNodIKDRyJTFDPjCbm5SrIq70hnKmH1UiLWz991Al7Ib/MJI9PjPQWzaYZZOtTq"
    "vLxLf1NnddlqlU9GaMfFYiydQFomJll+jqZCUFqdDSWrFVHqFsxO2A5xmLQRKCZAKYFJgAoCs1sX0M3QrjMKHUlm"
    "cHENeCg+TLt6rAH3JAizwVh7m4w1a1SPNeBgBGH8qgQhiQbC9y1dCH9rNhUbnDvI3C1hATFQfzcSBfW4t7qDjcQM"
    "QuZc+AMeSRwwsN8gRcCbUMzihOL2kmWUpsEyHJXBlyFzA6LSAneqYEelATTk0urFA3F6U2VFRiX43oilctpZTCqc"
    "OqUSQlhn5Sq1HXC8hQ494SkOzGVc2vktDsM/Uj7s1A5GeQio9+5uoxkw77q5hmp9AY26cVarC19bXKmiLMFsTPxA"
    "BklYGCxtu+RBASpd40AUtwQTKjbRGCSA3bqw23Pl95/LEgBKDmrbwVDvBlkZgjSDbdbiZ7MVSvwQlEywaq8ClpU9"
    "YuC0I3r4cKpSZVew/WelSSqndEpqEEu7ZDpiNcPEO84f7ZIh1HOZUHnhR1q7S1DJVEEzo+Tm2eoePkFOENc5EGeB"
    "YK51QYvdXiSUQ0YlAGk3QdUac+tQT3RpZ/xgGJZSgCFCS0zi3lCQQ/z8iLVXS8zucdLEYJ0MqKO4uyFjG4TxNtub"
    "9q2Qpc3qDgMW2PACtQ8v1MOwF0mIVR5xQ9NOxbAW1NxAyUrUKS5o7SxuNjyDk4+zV2OcO3Hra1aAElsVw6IIFqvc"
    "xBWzOlExJIxdTjEEIJPQuFECdQJrDL/rcgrJRVZUIuvcwEclf1l/uvJYLAs4o70qg2VzGdn7F2R1O61mdTsNF1bI"
    "BRU2b1rjwQpVFKgaOmddExAzgJSzr2sk2r06etMvD7H0poW70vj1ahm/XmCfzPe2bk2AFYamKh/T9W7yG8iUQ958"
    "0H+FScv94WH/AFKv4/HwRQ6ZmgVggpVLjlysvJHLQC9gd1w8GjIkCXaP8QxWLxAIj3bUrupIQ3Zq4+wG0xfvxSo5"
    "Ja1hfr4m276EhQWF1a6OZOkEMC4qV8AbGzeBb3PJN3A6LuNZK5w80I01XDtSwh3G2qmJtYpn/vzPpH8g3jLm+6Yr"
    "BhiIkpuQYS8YI7e+Z6GdI6w4qjpbc9h2aAcZge1E67ZgqyLDwuiNqLCoLMYhPmlg2xTxEkK0u6ATboU7iqHSM10G"
    "tWTMwhGXkJpFy9DqNe9FR7peynAM6HLTwg8eey0YcDOEnvy0CPpWCH1pi4AQ6GJw4PB7TBSCyr11qAuok5otS9j2"
    "AdrrakLDdF52BvlI8s4Xpk5eDpMdE1DF83w97wSitsrjUATuBG0RbaWVc4Jm37eB4R7bfhjBwsY69sMIQU5KT/VW"
    "S2eiXQC1iUHHSbgLehfj6GVcKpTrnHmbW9Ur2cUQATG6bqXba+Z7J2R4gmChLfop11k7GBn14jsujO++gOzfGQg/"
    "FupBNP14jooPo9uKUWMD7oehS8GzzbC3N8PeDZ/dYRAhLnoQoQi03pWrYyPKibUKAkAQaNupj5hhpam3NNJgVhFJ"
    "mWPZgaHIt3YVpyAcPI6Rj1LUqU9RMKQENuMMwwQGMBBqDgJmwZyVChe8E4LOlksWcqujwK0wZgaRhtExiKC/gPEg"
    "tNHKjJ+qSvsbZ6ODYiPPbdmlCCw6Abr/cqWrEyCMogtxNSW9Slp3YuN15SCEv92swu+b5jpoK1kcsMZsMtkPiiOd"
    "B0KWqJgXeJTyc1C5o+eJJnRZCgPHXGdCxZzzR4W9LCSisEdhvD2rZXusEZfuLZuNkDV3pR+g0rKTBFDwd69D7wAe"
    "9wqD4J3A2ehr9t2LTDvf/UzrEo8aXUgvOTCiZsCCVbbxbdpMH+eF74FcqvvdtQkgZ1TVaXflbcXioO2N5ibgS5SC"
    "98rnprlTPjchX6KKz63m5nPTaoXmJs61wF63HD7OZekxq0o1YZOWKuJHB3HNavOLw8z2whyyAcCd8lPS6ng0p8F0"
    "E9gBSyfHHJg2p4R1q4CJU65R6JGdi3lHB6zOySRBTQiF7WQiX7N0iuftKZJhnduVQFTDLY+WYQNGXi90dl/jsgHd"
    "yupgpyx0qyOH5iO/QkEeRKIY9aV4SbMLx5tQU5MrcrU2pzff0rFxKwbabAQyoibJBp/b4Yg6BH/U8TCYQIy+mt7x"
    "pJW+/AF3JXAGxzoz5N4I8RbPWton1j4zpOiCQrfyQoeK3mWkeUlVHovVH9E/sWmQw6D8zScxvz8W8uICOjEG1hST"
    "MvyH7Nft3iI/QIw+iKWJ4VyRrgD7wMOh6myrf4nG1JVyFCw8h6s7loNdTynRZfnVGqr6DLI9XoswVlDDJFAfczWZ"
    "amStzxt1rA5GToy2GEOzpMZqKu6dk2jziFQFZ4OcGXYbATnwhoDqvihTqNIRRuVG2YDBjkoQxswAB605KRncPMDT"
    "yBDqTpzT3DqWzmcuLIJzlsE0bMUAgzckYYCwDAq2dFdkHKSPCXbXhJ6N3JtcrYmaYKEEpVNtdO8oBTWHjHeYgKCI"
    "ShGEwAvunGOycV6Oxp+VSjwaWKmnL5AgW/g2kLiqkWjlyOaldExMEakYA82w4bA7K5ZqCvzWI9D6W8+V3YWjTniD"
    "kMw5ghIEfFfry41DzTtH+QQA1BotQRsepG5ZxkKsojq3Hj7BdU54nxG7zAnrGFYWk0FRIlfIEN/IfjFIDAdoFcus"
    "uwbKFlZQ/gxsUO85k+drVehAhTCoGzO+uosorokoGEbHv6+i0lVKj9Qi8lYWrlKdVS+rVm29KqtGyJdy57bMcntu"
    "mnaTAvo5pJI2XpgR1wwjnmI8f3KuOOG2pFqj+ILueSbeYo9YMkpK2QPmdTdYWRdzMkw7HBvMsyDnpA51pWz/8ulU"
    "/iRdwhXqucRLrdSfTuqz+oUsyJIXFqjjmOYRQ49ndKJCnzORRyywWpk3iyWvdepaZhb9xLWbro7nwQM/zWlX/hj0"
    "rN25ttkvJxFqTm9oqxTkNSwDEEyg4pMYv9ZdMlQP4HTFRpl+pbof38S816mutapfWNu5IPuxKhsYa7ZUn+/YybqF"
    "zHRRTHnO4aSme80O1kgrAyUm8uSuhtNbxgpgKdgKt2ws2sxo9rR7EsHDLjTkNBqq9cMFICCTGRt6uKGGxiOSztHK"
    "jX7KxB3OEbe0FyB1oQ8bsZyxj4e/rO/q4kXlkCMTKD0QfQStQyaSsvLqyMxH+1FQeEZBQf4YgbUJVmkMdvpDVUii"
    "z8XOdWNc4DW/Z0klZs/k/VGi/Y1T0+yXNTkOwNSKmb9Vd/mpSDlUtaGlgltX5LDwCoxzfpei9JXsxngCVTb3PlY7"
    "CVhaJ+9OMbup0ktUZDgGc9PaXUYtjkHA1z4Anh21N/bk28jdPQYrMQuKexXMHn0gFZew5IJ9x85nMXefrMuuzMU/"
    "/4ElgbPiFm2Btum4LTQ+4IKFSk3vcHchc+7sbf+UuQZuuynWVZ4yqrG4GBN5StvA1RvlN1fI+yrwugr2KG+u2K66"
    "uEB9foxVc7Gn6A0xcLw+2R+NX/bHA/H7cTI53nuW9CfJYf4y2RPgz+CWnEF+MNwbjo4nAnQ0eEwE5pOj7XIvUd2t"
    "ppeFzhOaKA8WWOIiUzsN5343tRVhHnQoVoH3KIF+NMbTOPPnZlIDr0SLG+E+nNiPBHhphMz11UAX64tZ2aeSq3Dw"
    "EpYkeqVMTVFxrzYxvJK9/4QzDrhe9A+O8+3KbecUr9la+M64Nb21F4g76tKBPvYXSrk5/UYyQTJg25VHW4s6slqx"
    "Oaohy1fkhXKumD3pja+q+Q1XXCqvHkLc1qzqj/O/FIf/D0XJ33bBWYrb/4+l9pOwpHIp6Y0jVaPH1CqpQLkhLeae"
    "cjRffBHh7LA5KWXTZfX//Ev9d7wCiR/QDdwPJzdNuopQb6KsR+Y6lrLOlgIgJSAX8Prv2V8pJ/DzN+yvlJtf4L9O"
    "aRHAOK+wDBXei3mimwHh2BUBOn/lDhOezV1SVa/1QVD35qMvssie8qkRqvPtL++ZyUFILGIr1Z8jZ1pKly/A/von"
    "XMR6AZMmC6ztbb6R4LJuX2pKx/Dg/Ip1dsVE+tTNnCoLqmYBnNtAaMrriW7AxDN9OlHMezT336se3WAXLftfioFu"
    "rrHtLYdzga3YvKz+aF254/AmeBmPdw2mungfdQJeA3EKRv2TDGFA9lqA4MEGPBWpb/+UL6E2X+9HKd9ER/uK/yx+"
    "FKT8yFNLHBNrQFuSABxDbg/ui8bivoxcDGoUqPe/MZB/TIJIX5BarTFIPOcqtmSuk/qAd6gWfxJtf3TqQ/g5KNjr"
    "4+WsEQZZE+GczFTxbx1TNNvCFR7p1FcBq+vPdTGbTt7Z+3T3Cgbw8iU/l/J/c3Gtw8M6+K6lp+xi1gXoVXa5w0xX"
    "85ELNytuZGrODutbQXGWp+D00IP6I0MM8cUrMOKZHdJB+n/nISsu5A4Zpwt5BwNGOJYzOaXZPhEiR5lXvBAv+oFu"
    "X0ESICKr7jB3T3ipidHfYq9UrN5HawpibihcYfQaT/ojh/Wo5N3ZQryabRnZgYV4iSee4x8p7DPFK8JYUETy7R7n"
    "2dRUW5Q5+h+l/L1ad9wS+IPQwlGYuJpUhmbg7AO/vMQCuZeCqNTuWSipxGVHMeyNbISIrW6dygolP2xCFni6Z6aO"
    "+6vioAiNNcmhmjEIudCNZVqs9caI1Qth9/KktfsdQjgQ7ZR7fFVJjroJ5xRnBi/PPqMDK0CNTscxV1u7SNiaqq7o"
    "Zm3ZE5194qc7FsqkqYwA6OtH//y/GM9FsVBpAAA="
)

def load_embedded():
    return json.loads(gzip.decompress(base64.b64decode(SESSION_B64)).decode("utf-8"))

SESSION = load_embedded()
print({k: v for k, v in SESSION.items() if k != "transcript"})

{'session_id': '8938cc17-0fe4-4323-a4f3-4af3d70d5fa5', 'source': 'coaching_sessions', 'stratum': 'coaching', 'user_id': '9a7b8b8e-87b1-473f-982d-c07c2cd1b53e', 'language': 'ur', 'db_duration_seconds': '1617', 'ffprobe_duration_seconds': '1617.46', 'audio_file': 'audio_files_v3/coaching/8938cc17-0fe4-4323-a4f3-4af3d70d5fa5.ogg', 'created_at': '2026-07-18T10:22:46.358000+00:00'}


In [9]:
#@title (optional) Upload a different session JSON — skip to keep the embedded one
UPLOAD = False   #@param {type:"boolean"}

if UPLOAD:
    from google.colab import files
    up = files.upload()
    fname = next(iter(up))
    SESSION = json.loads(up[fname].decode("utf-8"))
    print("Loaded", fname, "| session_id:", SESSION.get("session_id"))
else:
    print("Using embedded session:", SESSION.get("session_id"))

Using embedded session: 8938cc17-0fe4-4323-a4f3-4af3d70d5fa5


In [10]:
#@title Transcript stats
import re

TRANSCRIPT = SESSION["transcript"]
SESSION_META = {
    "session_id": SESSION.get("session_id", "?"),
    "language": SESSION.get("language", "?"),
    "duration_min": round(float(SESSION.get("ffprobe_duration_seconds",
                          SESSION.get("db_duration_seconds", 0)) or 0) / 60, 1),
    "recorded": SESSION.get("created_at", "?"),
}

turns = re.findall(r"\[(\d\d:\d\d)\]\s*([A-Za-z]+)\s*\(([A-Z]{2})\):", TRANSCRIPT)
speakers = pd.Series([t[1] for t in turns]).value_counts()
langs    = pd.Series([t[2] for t in turns]).value_counts()

print(json.dumps(SESSION_META, indent=2))
print(f"\nchars {len(TRANSCRIPT):,} | words {len(TRANSCRIPT.split()):,} | tagged turns {len(turns)}")
print("\nturns by speaker:\n", speakers.to_string())
print("\nturns by language:\n", langs.to_string())
print("\n--- first 600 chars ------------------------------------------------")
print(TRANSCRIPT[:600])

{
  "session_id": "8938cc17-0fe4-4323-a4f3-4af3d70d5fa5",
  "language": "ur",
  "duration_min": 27.0,
  "recorded": "2026-07-18T10:22:46.358000+00:00"
}

chars 18,237 | words 3,363 | tagged turns 264

turns by speaker:
 Student    132
Teacher    132

turns by language:
 UR    245
EN     19

--- first 600 chars ------------------------------------------------
[00:00] Student (EN): Go.

[00:02] Teacher (UR): ہاں جی، السلام علیکم۔ میرا موبائل یہاں پڑا رہے گا، ٹھیک ہے؟ السلام علیکم۔

[00:08] Student (UR): وعلیکم السلام۔

[00:10] Teacher (UR): ابھی جو پیپر سے پہلے چھٹیاں ہوئی تھیں، اس میں کون سا، آہ، فنکشن آیا تھا؟ کیا آیا تھا؟

[00:17] Student (UR): ریزلٹ۔

[00:18] Teacher (UR): رزلٹ سے پہلے کیا آیا تھا؟

[00:20] Student (UR): عید۔

[00:20] Teacher (UR): عید آئی تھی۔ عید کیسے منائی تھی آپ لوگوں نے؟

[00:23] Student (UR): بہت اچھی۔

[00:25] Teacher (UR): بہت اچھی۔ مزہ آیا تھا؟

[00:27] Student (UR): Yes.

[00:28] Teacher (UR): کیا کیا کیا تھا عید میں؟ 


---
## 5 · Load the model (T4-safe)

T4 is Turing (compute 7.5): **no bfloat16, no flash-attention**. So everything runs
`float16` with `sdpa` attention — except Gemma-2, which needs `eager` because of its
attention logit soft-capping. 4-bit NF4 + double quantisation keeps an 8–9B model around
5–6 GB, leaving ~9 GB of the T4's 15 GB for the KV cache — which matters here, because an
Urdu transcript tokenises expensively.

In [11]:
#@title Load tokenizer + model
import torch, gc, time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

REPO = MODEL_SPEC["repo"]
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(REPO, token=HF_TOKEN or None)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kw = dict(device_map="auto", token=HF_TOKEN or None,
               attn_implementation=MODEL_SPEC["attn"], low_cpu_mem_usage=True)

if MODEL_SPEC["quant"] == "4bit":
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
elif MODEL_SPEC["quant"] == "8bit":
    load_kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)

# transformers renamed torch_dtype -> dtype in 4.56. Pick the right key by version rather
# than try/except: older versions swallow an unknown `dtype` into **kwargs instead of raising,
# which would silently load in fp32 and OOM the T4.
from packaging.version import Version
_dtype_kw = ({"dtype": torch.float16}
             if Version(transformers.__version__.split("+")[0]) >= Version("4.56.0")
             else {"torch_dtype": torch.float16})
model = AutoModelForCausalLM.from_pretrained(REPO, **_dtype_kw, **load_kw)

model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id

MODEL_CTX = min(MODEL_SPEC["ctx"], getattr(model.config, "max_position_embeddings", MODEL_SPEC["ctx"]))
MODEL_CTX_MAX = MODEL_CTX
if CFG.MAX_CTX_TOKENS:                      # what the GPU can prefill, not what the model allows
    MODEL_CTX = min(MODEL_CTX, CFG.MAX_CTX_TOKENS)
print(f"\nloaded in {time.time()-t0:.0f}s | usable ctx {MODEL_CTX:,} tokens"
      + (f" (capped from {MODEL_CTX_MAX:,})" if MODEL_CTX < MODEL_CTX_MAX else ""))
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


loaded in 319s | usable ctx 131,072 tokens
GPU memory allocated: 5.71 GB / 15.6 GB


In [12]:
#@title chat() — the single generation entry point; every sampling knob comes from CFG
from contextlib import nullcontext

def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

# --- attention backend ----------------------------------------------------
# T4 is sm75. PyTorch's flash SDPA kernel needs sm80, so SDPA quietly falls back to the
# MATH backend, which materialises the full (1, n_heads, L, L) score matrix: at L=17k that
# is 32 x 17k x 17k x 2 bytes = 17.5 GB on a 14.5 GB card. Ask for the cutlass
# memory-efficient kernel first (O(L) memory); keep math only as a last resort.
try:
    from torch.nn.attention import sdpa_kernel, SDPBackend
    _BACKENDS = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
    def attn_ctx():
        try:
            return sdpa_kernel(_BACKENDS)                        # torch >= 2.5 takes a list
        except TypeError:
            return sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION)   # torch 2.4 takes one
except ImportError:
    def attn_ctx():
        return nullcontext()

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_free_gb():
    if not torch.cuda.is_available():
        return float("nan")
    free, _ = torch.cuda.mem_get_info()
    return free / 1e9

@torch.inference_mode()
def chat(system, user, seed, max_new_tokens=None, temperature=None):
    """One chat completion. Seeded so a given (prompt, seed) is reproducible."""
    temp = CFG.TEMPERATURE if temperature is None else temperature
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": user}]
    try:
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:      # Gemma has no system role -> fold it into the user turn
        merged = [{"role": "user", "content": (system + "\n\n" + user) if system else user}]
        prompt = tokenizer.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)

    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    n_new = max_new_tokens or CFG.MAX_NEW_TOKENS
    n_in  = enc["input_ids"].shape[1]
    if n_in + n_new > MODEL_CTX:
        raise ValueError(
            f"prompt is {n_in:,} tokens + {n_new:,} to generate > usable ctx {MODEL_CTX:,}. "
            f"Lower CFG.MAX_NEW_TOKENS / CFG.DIGEST_CHUNK_TOKENS, or raise CFG.MAX_CTX_TOKENS "
            f"if the GPU can take it (prefill memory grows with the SQUARE of the prompt).")

    set_seed(seed)                                    # <- reproducible sampling
    gen_kw = dict(max_new_tokens=n_new,
                  repetition_penalty=CFG.REPETITION_PENALTY,
                  pad_token_id=tokenizer.pad_token_id,
                  use_cache=True)
    if temp and temp > 0:
        gen_kw.update(do_sample=True, temperature=temp, top_p=CFG.TOP_P)
        if CFG.TOP_K:
            gen_kw["top_k"] = CFG.TOP_K
    else:
        gen_kw.update(do_sample=False)

    try:
        with attn_ctx():
            out = model.generate(**enc, **gen_kw)
    except torch.cuda.OutOfMemoryError:
        free_gpu()                                    # fragmentation, maybe. One honest retry.
        try:
            with attn_ctx():
                out = model.generate(**enc, **gen_kw)
        except torch.cuda.OutOfMemoryError as e:
            free_gpu()
            raise torch.cuda.OutOfMemoryError(
                f"OOM on a {n_in:,}-token prompt ({gpu_free_gb():.1f} GB free after clearing "
                f"the cache). Attention prefill is O(L^2): halve the prompt and you quarter the "
                f"peak. Fixes, cheapest first: lower CFG.MAX_CTX_TOKENS (forces the digest "
                f"strategy), lower CFG.DIGEST_CHUNK_TOKENS, or switch MODEL_KEY to a smaller "
                f"model. Re-run the CONFIG cell, then §6 onwards.\n\n{e}") from e
    finally:
        del enc

    return tokenizer.decode(out[0, n_in:], skip_special_tokens=True).strip()

# smoke test
print(chat("You are a terse assistant.", "Reply with exactly: OK", seed=0, max_new_tokens=8))
print(f"\ntranscript = {n_tokens(TRANSCRIPT):,} tokens under this tokenizer "
      f"({len(TRANSCRIPT)/max(n_tokens(TRANSCRIPT),1):.2f} chars/token — "
      f"Urdu script is expensive)")
print(f"GPU free after smoke test: {gpu_free_gb():.1f} GB")


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


OK

transcript = 10,807 tokens under this tokenizer (1.69 chars/token — Urdu script is expensive)


---
## 6 · Fitting the transcript in context

Urdu tokenises at roughly 1.3–1.8 chars/token, so this 18k-character transcript can exceed
Gemma-2's 8k window once the rubric and the completion are accounted for. `CONTEXT_STRATEGY`
handles it:

| Strategy | What happens |
|---|---|
| `full` | Whole transcript in the prompt. Best fidelity. Needs `MODEL_CTX` headroom. |
| `truncate` | Head + tail kept, middle dropped with an explicit marker. Fast, but the middle of a lesson is where the group work lives — expect this to depress C9/D3. |
| `digest` | Chunk → model writes a factual English evidence digest per chunk → score on the concatenated digest. Keeps whole-lesson coverage inside a small window. |
| `auto` | `full` if it fits, else `digest`. **Default.** |

### Why `MAX_CTX_TOKENS` exists

A model's advertised context is not a context your GPU can *prefill*. Attention memory during
prefill grows with the **square** of the prompt: on a T4 (sm75) PyTorch has no flash kernel, so
SDPA can end up materialising an `(1, n_heads, L, L)` score matrix — at L = 17k that is
`32 x 17k x 17k x 2 bytes` ≈ **17 GB**, on a card with ~6.5 GB free after the 4-bit weights.
Llama-3.1's 131k window therefore makes `auto` choose `full`, and `full` OOMs.

`MAX_CTX_TOKENS` caps the window the budgeting in this section is allowed to assume
(default **10,240**), which is what pushes `auto` onto `digest` for this transcript. Raise it
only on an A100/L4; set it to `0` to use the model's real maximum.

`DIGEST_ONCE=True` builds the digest **once** and reuses it for every iteration. That is
deliberate: it holds the input fixed so the wobble you measure is *scoring* wobble only.
Set it `False` to measure the compounded digest+scoring wobble (which is what a production
pipeline would actually experience).

In [13]:
#@title Build the scoring context
import re, time

def split_turns(transcript):
    """Split on timestamped turn boundaries, keeping the timestamps."""
    parts = re.split(r"(?=\[\d\d:\d\d\])", transcript)
    return [p.strip() for p in parts if p.strip()]

def chunk_by_tokens(transcript, max_tok):
    chunks, cur, cur_tok = [], [], 0
    for turn in split_turns(transcript):
        t = n_tokens(turn)
        if cur and cur_tok + t > max_tok:
            chunks.append("\n\n".join(cur)); cur, cur_tok = [], 0
        cur.append(turn); cur_tok += t
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks

DIGEST_SYSTEM = (
    "You are a classroom-observation analyst. You summarise lesson transcripts into factual "
    "evidence notes for a coaching rubric. You never praise, never judge, never score. "
    "You report only what is observable in the excerpt.")

DIGEST_USER = """Below is an excerpt from a classroom lesson transcript (Urdu/English, Pakistan primary school).

Write compact factual evidence notes in ENGLISH under exactly these headings. Use short bullets. If there is no evidence for a heading, write "none observed". Do not evaluate or score.

- OBJECTIVE/FRAMING: any statement of what is being learned, lesson/topic naming, agenda
- SEQUENCE: phases, transitions, recap, closure, homework
- TEACHER QUESTIONS: the actual questions asked, and their type (recall / open / why-how)
- TEACHER EXPLANATIONS: content taught, definitions given, any error or imprecision
- FEEDBACK: exact praise/correction phrases used, and whether specific
- PARTICIPATION: how many distinct students speak, named students, whole-class vs individual, gender if inferable
- GROUPING: pair/group/peer work instructions and whether students actually do it
- MATERIALS/TECH: textbook, board, worksheet, video, app, phone, manipulatives
- STUDENT TALK: what students actually say (length, language, whether content-related)
- PRIOR KNOWLEDGE / REAL WORLD: links to earlier lessons or students' own lives
- MANAGEMENT/TONE: routines, disruptions, warmth, punitive language
- TIME MARKERS: first and last timestamp in this excerpt, and any long off-task stretch

EXCERPT ({idx}/{total}, {start}):
\"\"\"
{chunk}
\"\"\""""

def build_digest(seed, verbose=True):
    chunks = chunk_by_tokens(TRANSCRIPT, CFG.DIGEST_CHUNK_TOKENS)
    notes = []
    for i, ch in enumerate(chunks, 1):
        m = re.match(r"\[(\d\d:\d\d)\]", ch)
        start = m.group(1) if m else "?"
        if verbose:
            print(f"  digest chunk {i}/{len(chunks)} (from {start}, {n_tokens(ch)} tok) ...", end="", flush=True)
        t = time.time()
        out = chat(DIGEST_SYSTEM,
                   DIGEST_USER.format(idx=i, total=len(chunks), start=start, chunk=ch),
                   seed=seed + i, max_new_tokens=520, temperature=min(CFG.TEMPERATURE, 0.2))
        notes.append(f"=== EVIDENCE FROM MINUTE {start} (part {i}/{len(chunks)}) ===\n{out}")
        if verbose:
            print(f" {time.time()-t:.0f}s")
    return "\n\n".join(notes)

def truncate_transcript(budget_tok):
    turns = split_turns(TRANSCRIPT)
    head, tail, ht, tt = [], [], 0, 0
    i, j = 0, len(turns) - 1
    while i <= j:
        if ht <= tt and ht + n_tokens(turns[i]) < budget_tok * 0.6:
            head.append(turns[i]); ht += n_tokens(turns[i]); i += 1
        elif tt + n_tokens(turns[j]) < budget_tok * 0.4:
            tail.insert(0, turns[j]); tt += n_tokens(turns[j]); j -= 1
        else:
            break
    dropped = j - i + 1
    return ("\n\n".join(head)
            + f"\n\n[... {dropped} turns omitted from the middle of the lesson ...]\n\n"
            + "\n\n".join(tail))

# ---- decide the strategy -------------------------------------------------
RUBRIC_BUDGET = max(n_tokens(render_section_rubric(s)) for s in CFG.SECTIONS)
AVAILABLE = MODEL_CTX - CFG.MAX_NEW_TOKENS - RUBRIC_BUDGET - CFG.CTX_HEADROOM
TRANSCRIPT_TOK = n_tokens(TRANSCRIPT)
strategy = CFG.CONTEXT_STRATEGY
if strategy == "auto":
    strategy = "full" if TRANSCRIPT_TOK < AVAILABLE else "digest"

print(f"model ctx {MODEL_CTX:,} | rubric {RUBRIC_BUDGET:,} | completion {CFG.MAX_NEW_TOKENS:,} "
      f"| headroom {CFG.CTX_HEADROOM:,}")
print(f"=> {AVAILABLE:,} tokens available for the transcript; transcript is {TRANSCRIPT_TOK:,}")
print(f"=> strategy: {strategy.upper()}\n")
if strategy == "full":
    # crude but load-bearing: peak prefill on the MATH attention backend is
    # n_heads x L^2 x 2 bytes. Warn before it OOMs 40 minutes into a run.
    _heads = getattr(model.config, "num_attention_heads", 32)
    _peak  = _heads * (TRANSCRIPT_TOK + RUBRIC_BUDGET) ** 2 * 2 / 1e9
    if _peak > gpu_free_gb() * 0.6:
        print(f"WARNING: a ~{TRANSCRIPT_TOK + RUBRIC_BUDGET:,}-token prompt can need up to "
              f"{_peak:.0f} GB of attention scratch if SDPA falls back to the math kernel, "
              f"against {gpu_free_gb():.1f} GB free. Set CFG.MAX_CTX_TOKENS to force the "
              f"digest strategy if this OOMs.\n")

if strategy == "full":
    SCORING_CONTEXT = TRANSCRIPT
    CONTEXT_KIND = "verbatim transcript"
elif strategy == "truncate":
    SCORING_CONTEXT = truncate_transcript(AVAILABLE)
    CONTEXT_KIND = "verbatim transcript (middle truncated)"
else:
    print("building evidence digest ...")
    t0 = time.time()
    SCORING_CONTEXT = build_digest(CFG.BASE_SEED)
    CONTEXT_KIND = "structured evidence digest of the full transcript"
    print(f"digest built in {time.time()-t0:.0f}s")

CONTEXT_TOK = n_tokens(SCORING_CONTEXT)
print(f"\nscoring context: {CONTEXT_KIND} — {CONTEXT_TOK:,} tokens")
assert CONTEXT_TOK < AVAILABLE + CFG.CTX_HEADROOM, \
    "context still too long: lower DIGEST_CHUNK_TOKENS or MAX_NEW_TOKENS, or pick a longer-ctx model"
print("\n--- context preview ----------------------------------------------")
print(SCORING_CONTEXT[:1200], "...")

model ctx 131,072 | rubric 1,347 | completion 1,100 | headroom 1,600
=> 127,025 tokens available for the transcript; transcript is 10,807
=> strategy: FULL


scoring context: verbatim transcript — 10,807 tokens

--- context preview ----------------------------------------------
[00:00] Student (EN): Go.

[00:02] Teacher (UR): ہاں جی، السلام علیکم۔ میرا موبائل یہاں پڑا رہے گا، ٹھیک ہے؟ السلام علیکم۔

[00:08] Student (UR): وعلیکم السلام۔

[00:10] Teacher (UR): ابھی جو پیپر سے پہلے چھٹیاں ہوئی تھیں، اس میں کون سا، آہ، فنکشن آیا تھا؟ کیا آیا تھا؟

[00:17] Student (UR): ریزلٹ۔

[00:18] Teacher (UR): رزلٹ سے پہلے کیا آیا تھا؟

[00:20] Student (UR): عید۔

[00:20] Teacher (UR): عید آئی تھی۔ عید کیسے منائی تھی آپ لوگوں نے؟

[00:23] Student (UR): بہت اچھی۔

[00:25] Teacher (UR): بہت اچھی۔ مزہ آیا تھا؟

[00:27] Student (UR): Yes.

[00:28] Teacher (UR): کیا کیا کیا تھا عید میں؟ نئے کپڑے بنائے تھے؟

[00:33] Student (UR): Yes.

[00:34] Teacher (UR): فرینڈز کے گھر گئے تھے؟

[00:36] Student (UR): Yes.

---
## 7 · Scoring prompts & parsing

The prompt is deliberately austere: rubric + evidence + "return JSON". No few-shot examples,
no self-consistency voting, no ensembling — those all *suppress* wobble, and suppressed wobble
is exactly what we are trying to measure. What you get here is the honest baseline.

Three parse guards, in order: strict JSON → brace-matched JSON substring → per-code regex
(`"B3"… 3`). Anything that survives all three is recorded as a **parse failure** (`NaN`), and
the failure rate is reported as its own wobble metric — a model that can't hold format is
wobbly in a way SD alone won't show.

**On `NA`:** F5 (Math) and F6 (Science) do not apply to an English reading lesson. With
`ALLOW_NA=True` the model may return `"NA"`, recorded as `NaN` and excluded from that
indicator's statistics; the **NA rate** is reported separately, because *inconsistent
applicability judgements are themselves wobble*. Set `ALLOW_NA=False` to force a 1–4 on
every indicator — the scores stay whole numbers 1–4 as specified, but expect F5/F6 wobble to
inflate, since the model is being made to score something that isn't there.

In [14]:
#@title Prompt builders
SCORING_SYSTEM = (
    "You are a trained classroom observer for Taleemabad, scoring a lesson against the "
    "Taleemabad Coaching Framework. You are strict, evidence-bound and calibrated:\n"
    "- Score ONLY on evidence present in the material provided. Absence of evidence is "
    "score 1, never a generous guess.\n"
    "- A score of 3 requires the level-3 descriptor to be clearly met, not partially.\n"
    "- Do not reward effort, warmth or busyness when the descriptor asks for something else.\n"
    "- Output valid JSON only. No preamble, no markdown fences, no commentary.")

_NA_CLAUSE = ('If an indicator genuinely cannot apply to this lesson (e.g. a MATH-specific '
              'indicator in a language lesson), use the string "NA" instead of a number.')

_JSON_SHAPE_EV = ('{{"{first}": {{"score": <1|2|3|4>, "evidence": "<max 20 words quoted or '
                  'paraphrased from the material>"}}, ...}}')
_JSON_SHAPE_NO = '{{"{first}": <1|2|3|4>, ...}}'

def build_scoring_prompt(section_code, codes, variant=None):
    variant = variant or CFG.PROMPT_VARIANT
    rubric  = render_section_rubric(section_code, codes=codes, terse=(variant == "terse"))
    shape   = (_JSON_SHAPE_EV if CFG.INCLUDE_EVIDENCE else _JSON_SHAPE_NO).format(first=codes[0])
    na      = ("\n" + _NA_CLAUSE) if CFG.ALLOW_NA else ""
    keys    = ", ".join(codes)

    task = {
        "standard": (
            f"Score EVERY indicator listed below on the 1-4 scale, using the level descriptors "
            f"as the definition of each score. Whole numbers only.{na}\n\n"
            f"Return exactly one JSON object with these {len(codes)} keys and nothing else: {keys}\n"
            f"Shape: {shape}"),
        "terse": (
            f"Score each indicator 1-4. Whole numbers.{na}\n"
            f"JSON only, keys: {keys}\nShape: {shape}"),
        "cot": (
            f"For each indicator: first weigh the evidence for and against each level in one "
            f"sentence, then commit to a whole-number score 1-4.{na}\n\n"
            f"Write your reasoning inside a single <reasoning>...</reasoning> block "
            f"(keep it under 25 words per indicator), then output the JSON object with keys "
            f"{keys} after the closing tag.\nShape: {shape}"),
    }[variant]

    user = (f"## MATERIAL TO SCORE\n"
            f"Source: {CONTEXT_KIND} of a lesson observation.\n"
            f"Session: {SESSION_META['session_id']} | language {SESSION_META['language']} | "
            f"{SESSION_META['duration_min']} minutes.\n\n"
            f'"""\n{SCORING_CONTEXT}\n"""\n\n'
            f"## RUBRIC\n{rubric}\n\n"
            f"## TASK\n{task}")
    return SCORING_SYSTEM, user

_s, _u = build_scoring_prompt("D", SECTION_CODES["D"])
print(f"prompt = {n_tokens(_s) + n_tokens(_u):,} tokens (section D)")
print("\n--- task tail ----------------------------------------------------")
print(_u[-900:])

prompt = 11,835 tokens (section D)

--- task tail ----------------------------------------------------
se.

### D7 — Inclusivity of Engagement
  1 (Not Observed / Emerging): Only front-row or high-ability students engaged.
  2 (Developing): Teacher attempts inclusion but success is limited.
  3 (Proficient / Effective): Students across ability levels and genders are participating.
  4 (Highly Effective): Deliberate inclusion of marginalized students; no one invisible. Gender-equitable participation.

## TASK
Score EVERY indicator listed below on the 1-4 scale, using the level descriptors as the definition of each score. Whole numbers only.
If an indicator genuinely cannot apply to this lesson (e.g. a MATH-specific indicator in a language lesson), use the string "NA" instead of a number.

Return exactly one JSON object with these 7 keys and nothing else: D1, D2, D3, D4, D5, D6, D7
Shape: {"D1": {"score": <1|2|3|4>, "evidence": "<max 20 words quoted or paraphrased from the material>"}, .

In [15]:
#@title JSON parsing with three fallbacks
import math

def _brace_slice(text):
    """Longest balanced {...} span in the text."""
    starts = [i for i, c in enumerate(text) if c == "{"]
    for s in starts:
        depth, in_str, esc = 0, False, False
        for i in range(s, len(text)):
            c = text[i]
            if in_str:
                if esc:            esc = False
                elif c == "\\":    esc = True
                elif c == '"':     in_str = False
                continue
            if c == '"':   in_str = True
            elif c == "{": depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    return text[s:i + 1]
    return None

def _coerce(val):
    """-> int 1..4, or None for NA / unparseable."""
    if isinstance(val, dict):
        val = val.get("score", val.get("value", val.get("rating")))
    if isinstance(val, bool):
        return None
    if isinstance(val, (int, float)):
        if isinstance(val, float) and math.isnan(val):
            return None
        v = int(round(val))
        return v if 1 <= v <= 4 else None
    if isinstance(val, str):
        s = val.strip()
        if s.upper() in ("NA", "N/A", "NONE", "NULL", "NOT APPLICABLE", ""):
            return None
        m = re.search(r"[1-4]", s)
        return int(m.group()) if m else None
    return None

def _evidence(val):
    if isinstance(val, dict):
        for k in ("evidence", "justification", "reason", "note"):
            if isinstance(val.get(k), str):
                return val[k][:300]
    return ""

def parse_scores(text, codes):
    """-> (scores {code: int|None}, evidence {code: str}, method str)"""
    scores  = {c: None for c in codes}
    ev      = {c: "" for c in codes}
    payload, method = None, "regex"

    body = re.sub(r"<reasoning>.*?</reasoning>", " ", text, flags=re.S | re.I)
    body = re.sub(r"^\s*```(?:json)?|```\s*$", " ", body.strip(), flags=re.M)

    for cand, name in ((body.strip(), "strict"), (_brace_slice(body), "brace")):
        if not cand:
            continue
        for attempt in (cand, re.sub(r",\s*([}\]])", r"\1", cand)):   # drop trailing commas
            try:
                payload, method = json.loads(attempt), name
                break
            except Exception:
                payload = None
        if payload is not None:
            break

    if isinstance(payload, dict):
        # tolerate {"scores": {...}} and {"B1": {...}} alike
        if len(payload) == 1 and isinstance(next(iter(payload.values())), dict) \
           and not set(payload) & set(codes):
            payload = next(iter(payload.values()))
        upper = {str(k).strip().upper(): v for k, v in payload.items()}
        for c in codes:
            if c in upper:
                scores[c] = _coerce(upper[c]); ev[c] = _evidence(upper[c])

    # fallback / gap-fill by regex
    if any(v is None for v in scores.values()):
        for c in codes:
            if scores[c] is not None:
                continue
            m = re.search(rf'["\']?\b{c}\b["\']?\s*[:\-=]\s*(?:\{{[^}}]*?["\']score["\']\s*:\s*)?'
                          rf'["\']?(NA|N/A|[1-4])', text, flags=re.I)
            if m:
                scores[c] = _coerce(m.group(1))
    return scores, ev, method

# self-test on messy output
_demo = """Sure! Here you go:
```json
{"D1": {"score": 3, "evidence": "most students read aloud"}, "D2": {"score": 2, "evidence": "rote repetition"},
 "D3": {"score": 2}, "D4": 3, "D5": "2", "D6": {"score": 1}, "D7": "NA",}
```"""
print(parse_scores(_demo, SECTION_CODES["D"]))

({'D1': 3, 'D2': 2, 'D3': 2, 'D4': 3, 'D5': 2, 'D6': 1, 'D7': None}, {'D1': 'most students read aloud', 'D2': 'rote repetition', 'D3': '', 'D4': '', 'D5': '', 'D6': '', 'D7': ''}, 'brace')


In [16]:
#@title score_section / run_iteration
import random

def score_section(section_code, iteration, verbose=None):
    verbose = CFG.VERBOSE if verbose is None else verbose
    codes = list(SECTION_CODES[section_code])
    if CFG.INDICATOR_ORDER == "shuffled":
        random.Random(CFG.BASE_SEED + 977 * iteration + hash(section_code) % 1000).shuffle(codes)

    groups = [codes] if CFG.SCORING_MODE == "per_section" else [[c] for c in codes]
    rows, raw_log = [], []

    for gi, group in enumerate(groups):
        system, user = build_scoring_prompt(section_code, group)
        got, ev, method, raw = {}, {}, "none", ""
        for attempt in range(CFG.MAX_RETRIES + 1):
            seed = CFG.BASE_SEED + 1000 * iteration + 37 * gi + attempt
            u = user if attempt == 0 else (
                user + "\n\nIMPORTANT: your previous reply was not parseable. Reply with the raw "
                       "JSON object ONLY - no prose, no fences, no trailing commas.")
            raw = chat(system, u, seed=seed,
                       max_new_tokens=CFG.MAX_NEW_TOKENS if len(group) > 1 else 260)
            got, ev, method = parse_scores(raw, group)
            if all(k in got for k in group) and not all(v is None for v in got.values()):
                break
        raw_log.append(raw)
        for c in group:
            rows.append(dict(iteration=iteration, section=section_code, code=c,
                             indicator=CODE2NAME[c], score=got.get(c),
                             evidence=ev.get(c, ""), parse=method,
                             na=(got.get(c) is None and method != "none")))
    if verbose:
        got_n = sum(r["score"] is not None for r in rows)
        print(f"    {section_code}: {got_n}/{len(rows)} scored  "
              f"[{' '.join(str(r['score'] if r['score'] is not None else '-') for r in sorted(rows, key=lambda r: CODE_ORDER[r['code']]))}]")
    return rows, raw_log

def run_iteration(iteration):
    rows, raws = [], {}
    for s in CFG.SECTIONS:
        r, raw = score_section(s, iteration)
        rows += r
        raws[s] = raw
    return rows, raws

---
## 8 · Run the evaluation

`N_ITERATIONS` independent passes over the same material with the same prompt. Iteration *i*
uses seed `BASE_SEED + 1000·i`, so the whole experiment is reproducible while each pass draws
a genuinely different sample.

Results are appended to `wobble_out/scores_long.csv` after every iteration — if Colab
disconnects mid-run, whatever finished is still on disk and the analysis below runs on it.

In [18]:
#@title ▶ RUN — scores every indicator, N_ITERATIONS times
import time, pathlib

records, raw_store = [], {}
t_start = time.time()

for it in range(CFG.N_ITERATIONS):
    t0 = time.time()
    free_gpu()          # KV cache from the previous iteration is dead weight
    print(f"[iteration {it+1}/{CFG.N_ITERATIONS}]  seed={CFG.BASE_SEED + 1000*it}")
    if not CFG.DIGEST_ONCE and strategy == "digest":
        SCORING_CONTEXT = build_digest(CFG.BASE_SEED + 1000 * it, verbose=False)   # re-digest per run
    rows, raws = run_iteration(it)
    records += rows
    raw_store[it] = raws
    pd.DataFrame(records).to_csv(f"{CFG.OUT_DIR}/scores_long.csv", index=False)
    dt = time.time() - t0
    print(f"    -> {dt:.0f}s  (elapsed {(time.time()-t_start)/60:.1f} min)"
          f"  [{gpu_free_gb():.1f} GB GPU free]", end="")
    print(f"  | projected total {dt*CFG.N_ITERATIONS/60:.0f} min" if it == 0 else "")

EXPECTED_CODES = [c for c in ALL_CODES if CODE2SECTION[c] in CFG.SECTIONS]

scores_long = pd.DataFrame(records)
scores_long["score"] = pd.to_numeric(scores_long["score"], errors="coerce")
scores_long["section"] = pd.Categorical(scores_long["section"], list(CFG.SECTIONS), ordered=True)
scores_long["code"]    = pd.Categorical(scores_long["code"], EXPECTED_CODES, ordered=True)

RUN_META = dict(**SESSION_META, model=REPO, quant=MODEL_SPEC["quant"],
                context_strategy=strategy, context_tokens=CONTEXT_TOK,
                **{k: v for k, v in asdict(CFG).items()})
with open(f"{CFG.OUT_DIR}/run_meta.json", "w") as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in RUN_META.items()}, f, indent=2)

print(f"\nDONE in {(time.time()-t_start)/60:.1f} min — {len(scores_long):,} score cells "
      f"({scores_long.score.notna().sum():,} numeric, "
      f"{scores_long.score.isna().sum():,} NA/failed)")
display(scores_long.head())

[iteration 1/10]  seed=1234


OutOfMemoryError: CUDA out of memory. Tried to allocate 17.46 GiB. GPU 0 has a total capacity of 14.56 GiB of which 6.58 GiB is free. Including non-PyTorch memory, this process has 7.98 GiB memory in use. Of the allocated memory 7.55 GiB is allocated by PyTorch, and 315.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [19]:
#@title Wide view — the raw wobble, before any statistics
# NOTE: reindexed against EXPECTED_CODES on purpose - an indicator the model returned NA
# for in *every* run must still appear as a row (na_rate = 1.0), not vanish from the report.
piv = (scores_long.pivot_table(index="code", columns="iteration", values="score",
                               observed=True, dropna=False)
       .reindex(index=EXPECTED_CODES, columns=range(CFG.N_ITERATIONS)))
wide = piv.set_axis(pd.MultiIndex.from_arrays(
    [[CODE2SECTION[c] for c in piv.index], list(piv.index)], names=["section", "code"]))
wide.columns = [f"run{c+1}" for c in wide.columns]
wide.insert(0, "indicator", [CODE2NAME[c] for _, c in wide.index])
wide["distinct"] = wide.filter(like="run").nunique(axis=1)
wide["range"]    = wide.filter(like="run").max(axis=1) - wide.filter(like="run").min(axis=1)
wide.to_csv(f"{CFG.OUT_DIR}/scores_wide.csv")

pd.set_option("display.width", 200, "display.max_columns", 40)
display(wide.style.background_gradient(cmap="Blues", subset=wide.filter(like="run").columns,
                                       vmin=1, vmax=4).format(precision=0, na_rep="NA"))
print(f"\nindicators that never changed across runs: "
      f"{(wide.distinct == 1).sum()}/{len(wide)}")

NameError: name 'scores_long' is not defined

In [ ]:
#@title Sample of the model's own evidence strings (sanity check on the scores)
if CFG.INCLUDE_EVIDENCE:
    ev = (scores_long[scores_long.evidence.astype(str).str.len() > 3]
          .sort_values("iteration").groupby("code", observed=True).tail(1)
          .set_index("code")[["score", "evidence"]])
    ev.insert(0, "indicator", [CODE2NAME[c] for c in ev.index])
    display(ev)
else:
    print("INCLUDE_EVIDENCE is off - no evidence strings collected.")

---
## 9 · Statistics

### 9.1 Per-indicator wobble

For each indicator, across the *N* runs:

| Column | Meaning |
|---|---|
| `mean`, `sd` | central tendency and wobble magnitude in rubric points |
| `ci_lo`, `ci_hi`, `ci_width` | percentile-bootstrap 95% CI of the mean (`N_BOOTSTRAP` resamples). `ci_width` is the practical read: an indicator whose CI spans 2.4–3.3 cannot be reported as "Proficient" |
| `mode`, `modal_share` | the score you'd report, and how often the model actually produced it |
| `range`, `distinct` | worst-case spread and how many different scores appeared |
| `entropy` | Shannon entropy over the four levels, normalised to 0–1 (0 = perfectly stable, 1 = uniform across all four) |
| `flip_rate` | decision instability at the Proficient line: `2·min(p, 1−p)` where `p = P(score ≥ 3)`. 0 = the pass/fail call never changes, 1 = coin flip |
| `na_rate` | share of runs returning NA / unparseable |
| `p_wobble` | one-sided exact binomial test, H0: *disagreement-with-mode rate ≤ `NEGLIGIBLE_DISAGREEMENT` (5%)*. Small p ⇒ **the wobble is statistically significant, not sampling luck** |
| `q_wobble` | `p_wobble` after Holm–Bonferroni correction across all indicators tested |
| `p_vs_random` | Monte-Carlo test, H0: *the model is scoring uniformly at random over {1,2,3,4}*. Small p ⇒ the model is genuinely more consistent than chance (a floor check — a "stable" indicator that fails this is stable for the wrong reason) |
| `grade` | stable / minor / material / severe (see thresholds in the cell) |

In [ ]:
#@title Statistical machinery
import numpy as np
from scipy import stats

RNG = np.random.default_rng(CFG.STATS_SEED)
LEVELS = np.array([1, 2, 3, 4])

# ---------- basic ----------
def boot_ci(x, stat=np.mean, n_boot=None, level=None, rng=None):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0:   return (np.nan, np.nan)
    if len(x) == 1:   return (float(x[0]), float(x[0]))
    n_boot = n_boot or CFG.N_BOOTSTRAP
    level  = level or CFG.CI_LEVEL
    rng    = rng or RNG
    draws  = rng.choice(x, size=(n_boot, len(x)), replace=True)
    vals   = stat(draws, axis=1)
    a      = (1 - level) / 2
    return tuple(float(v) for v in np.quantile(vals, [a, 1 - a]))

def t_ci(x, level=None):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    level = level or CFG.CI_LEVEL
    if len(x) < 2: return (np.nan, np.nan)
    se = stats.sem(x)
    if se == 0:    return (float(x.mean()), float(x.mean()))
    h = se * stats.t.ppf(0.5 + level / 2, len(x) - 1)
    return (float(x.mean() - h), float(x.mean() + h))

def norm_entropy(x, k=4):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return np.nan
    p = np.array([(x == lv).mean() for lv in LEVELS])
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(k))

_MC_CACHE = {}
def p_vs_random(x, sims=None):
    """H0: scores ~ Uniform{1,2,3,4}. p = P(SD_sim <= SD_obs)."""
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    n = len(x)
    if n < 2: return np.nan
    sims = sims or CFG.MC_SIMS
    key = (n, sims)
    if key not in _MC_CACHE:
        draws = np.random.default_rng(CFG.STATS_SEED + n).integers(1, 5, size=(sims, n))
        _MC_CACHE[key] = draws.std(axis=1, ddof=1)
    return float(((_MC_CACHE[key] <= x.std(ddof=1)) .sum() + 1) / (sims + 1))

def p_wobble_test(x, h0=None):
    """H0: P(run disagrees with the modal score) <= h0. One-sided exact binomial."""
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return np.nan, 0, 0
    h0 = CFG.NEGLIGIBLE_DISAGREEMENT if h0 is None else h0
    mode = stats.mode(x, keepdims=False).mode
    k, n = int((x != mode).sum()), len(x)
    return float(stats.binomtest(k, n, h0, alternative="greater").pvalue), k, n

def holm(pvals):
    p = np.asarray(pvals, float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m, running = len(idx), 0.0
    for i, j in enumerate(idx):
        running = max(running, min(1.0, (m - i) * p[j]))
        out[j] = running
    return out

# ---------- reliability ----------
def _delta2_ordinal(marg):
    """Ordinal difference function from coincidence marginals."""
    V = len(marg)
    cum = np.concatenate([[0.0], np.cumsum(marg)])
    d = np.zeros((V, V))
    for c in range(V):
        for k in range(V):
            lo, hi = min(c, k), max(c, k)
            s = cum[hi + 1] - cum[lo]                  # sum of n_g, g = lo..hi
            d[c, k] = (s - (marg[c] + marg[k]) / 2.0) ** 2
    return d

def krippendorff_alpha(data, level="ordinal"):
    """data: units x raters, NaN = missing. Units with <2 values are dropped."""
    data = np.asarray(data, float)
    obs = data[~np.isnan(data)]
    if obs.size == 0: return np.nan
    vals = np.unique(obs)
    V = len(vals)
    if V < 2: return 1.0                               # no disagreement possible
    ix = {v: i for i, v in enumerate(vals)}
    O = np.zeros((V, V))
    for row in data:
        r = row[~np.isnan(row)]
        m = len(r)
        if m < 2: continue
        cnt = np.zeros(V)
        for v in r: cnt[ix[v]] += 1
        for c in range(V):
            for k in range(V):
                O[c, k] += (cnt[c] * (cnt[k] - (1 if c == k else 0))) / (m - 1)
    marg = O.sum(axis=1)
    n = marg.sum()
    if n < 2: return np.nan
    d2 = _delta2_ordinal(marg) if level == "ordinal" else (1 - np.eye(V))
    Do = (O * d2).sum()
    E  = np.outer(marg, marg).astype(float)
    np.fill_diagonal(E, 0.0)
    De = (E * d2).sum() / (n - 1)
    return float(1 - Do / De) if De > 0 else np.nan

def fleiss_kappa(data, categories=LEVELS):
    """data: units x raters (complete rows only)."""
    data = np.asarray(data, float)
    data = data[~np.isnan(data).any(axis=1)]
    N, k = data.shape
    if N == 0 or k < 2: return np.nan
    C = np.array([[(row == c).sum() for c in categories] for row in data], float)
    Pi = ((C ** 2).sum(axis=1) - k) / (k * (k - 1))
    Pbar = Pi.mean()
    pj = C.sum(axis=0) / (N * k)
    Pe = (pj ** 2).sum()
    return float((Pbar - Pe) / (1 - Pe)) if Pe < 1 else np.nan

def icc21(data):
    """ICC(2,1) two-way random, single measure + F tests for units and for raters."""
    Y = np.asarray(data, float)
    Y = Y[~np.isnan(Y).any(axis=1)]
    n, k = Y.shape
    if n < 2 or k < 2: return dict(icc=np.nan)
    gm = Y.mean()
    SSR = k * ((Y.mean(axis=1) - gm) ** 2).sum()
    SSC = n * ((Y.mean(axis=0) - gm) ** 2).sum()
    SST = ((Y - gm) ** 2).sum()
    SSE = SST - SSR - SSC
    MSR = SSR / (n - 1); MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))
    denom = MSR + (k - 1) * MSE + k * (MSC - MSE) / n
    icc = (MSR - MSE) / denom if denom != 0 else np.nan
    out = dict(icc=float(icc), n_units=n, k_raters=k, MSR=MSR, MSC=MSC, MSE=MSE)
    if MSE > 0:
        out["F_units"]  = MSR / MSE
        out["p_units"]  = float(stats.f.sf(MSR / MSE, n - 1, (n - 1) * (k - 1)))
        out["F_raters"] = MSC / MSE
        out["p_raters"] = float(stats.f.sf(MSC / MSE, k - 1, (n - 1) * (k - 1)))
    else:                                              # perfectly consistent
        out.update(F_units=np.inf, p_units=0.0, F_raters=np.nan, p_raters=np.nan)
    return out

def pairwise_agreement(data):
    """Mean share of run-pairs giving the identical score, over units."""
    data = np.asarray(data, float)
    vals = []
    for row in data:
        r = row[~np.isnan(row)]
        if len(r) < 2: continue
        same = sum(1 for i in range(len(r)) for j in range(i + 1, len(r)) if r[i] == r[j])
        vals.append(same / (len(r) * (len(r) - 1) / 2))
    return float(np.mean(vals)) if vals else np.nan

def grade_wobble(modal_share, rng_, na_rate):
    if na_rate >= 0.5:                             return "severe"
    if modal_share >= 0.999 and rng_ == 0:         return "stable"
    if modal_share >= 0.80 and rng_ <= 1:          return "minor"
    if modal_share >= 0.60 and rng_ <= 2:          return "material"
    return "severe"

print("stats helpers ready")

In [ ]:
#@title Per-indicator wobble table
MATRIX = wide.filter(like="run").to_numpy(float)        # indicators x runs
IND_CODES = [c for _, c in wide.index]

# every row carries the full column set, so a run where nothing parsed still yields a
# well-formed (all-NaN) table instead of a KeyError three cells later
BLANK = dict.fromkeys(
    ["mean", "sd", "median", "mode", "modal_share", "min", "max", "range", "distinct",
     "ci_lo", "ci_hi", "ci_width", "t_lo", "t_hi", "entropy", "p_proficient", "flip_rate",
     "two_band_rate", "n_disagree", "p_wobble", "p_vs_random"], np.nan)

rows = []
for code_, xs in zip(IND_CODES, MATRIX):
    x = xs[~np.isnan(xs)]
    n_na = int(np.isnan(xs).sum())
    if len(x) == 0:
        rows.append(dict(section=CODE2SECTION[code_], code=code_, indicator=CODE2NAME[code_],
                         n=0, na_rate=1.0, grade="severe", **BLANK)); continue
    mode = int(stats.mode(x, keepdims=False).mode)
    modal_share = float((x == mode).mean())
    lo, hi = boot_ci(x)
    p_w, k_dis, n_dis = p_wobble_test(x)
    p_prof = float((x >= CFG.PROFICIENCY_CUT).mean())
    rng_ = float(x.max() - x.min())
    rows.append(dict(
        section=CODE2SECTION[code_], code=code_, indicator=CODE2NAME[code_],
        n=len(x), na_rate=n_na / len(xs),
        mean=float(x.mean()), sd=float(x.std(ddof=1)) if len(x) > 1 else 0.0,
        median=float(np.median(x)), mode=mode, modal_share=modal_share,
        min=float(x.min()), max=float(x.max()), range=rng_, distinct=int(len(np.unique(x))),
        ci_lo=lo, ci_hi=hi, ci_width=hi - lo,
        t_lo=t_ci(x)[0], t_hi=t_ci(x)[1],
        entropy=norm_entropy(x),
        p_proficient=p_prof, flip_rate=2 * min(p_prof, 1 - p_prof),
        two_band_rate=float((np.abs(x - mode) >= 2).mean()),
        n_disagree=k_dis, p_wobble=p_w, p_vs_random=p_vs_random(x),
    ))

ind_stats = pd.DataFrame(rows)
if ind_stats.n.sum() == 0:
    raise RuntimeError(
        "No indicator produced a single parseable score. Check the raw model output in "
        "raw_store[0] — the model is probably refusing the JSON format. Try "
        "PROMPT_VARIANT='terse', a lower TEMPERATURE, or a different MODEL_KEY.")
ind_stats["q_wobble"] = holm(ind_stats["p_wobble"].to_numpy())
ind_stats["sig_wobble"] = ind_stats["q_wobble"] < CFG.ALPHA
ind_stats["grade"] = [grade_wobble(r.modal_share, r.range, r.na_rate)
                      if r.n else "severe" for r in ind_stats.itertuples()]
ind_stats["section"] = pd.Categorical(ind_stats.section, list(CFG.SECTIONS), ordered=True)
ind_stats = ind_stats.sort_values(["section", "code"],
                                  key=lambda s: s.map(CODE_ORDER) if s.name == "code" else s)
ind_stats.to_csv(f"{CFG.OUT_DIR}/indicator_wobble.csv", index=False)

SHOW = ["section", "code", "indicator", "n", "mean", "sd", "mode", "modal_share",
        "range", "ci_lo", "ci_hi", "ci_width", "entropy", "flip_rate", "na_rate",
        "p_wobble", "q_wobble", "p_vs_random", "grade"]
display(ind_stats[SHOW].style
        .format({c: "{:.2f}" for c in ["mean", "sd", "modal_share", "range", "ci_lo", "ci_hi",
                                       "ci_width", "entropy", "flip_rate", "na_rate"]})
        .format({c: "{:.4f}" for c in ["p_wobble", "q_wobble", "p_vs_random"]})
        .background_gradient(cmap="Reds", subset=["sd", "ci_width", "entropy", "flip_rate"]))

### 9.2 Reliability & run-to-run drift

Three coefficients, because they answer different questions:

- **Krippendorff's α (ordinal)** — the right one for this design: ordered categories, handles the
  NA cells. Treats each run as a "rater" of the 37 indicators.
  Convention: **α ≥ 0.80** dependable, **0.67–0.80** tentative, **< 0.67** not usable for decisions.
- **ICC(2,1)** — how much of the score variance is real between-indicator signal rather than
  run noise. Its two F-tests are the useful part: `p_units` (do the indicators genuinely differ,
  i.e. is there signal at all?) and `p_raters` (**do runs systematically drift** — one pass
  scoring the whole lesson higher than another?).
- **Fleiss' κ** — nominal, ignores ordering, so it treats a 3-vs-4 slip as badly as 1-vs-4.
  Reported as the pessimistic bound.

Plus a **Friedman test** (non-parametric, no normality assumption) on runs × indicators as the
distribution-free check on drift, and **Levene's test** on within-indicator deviations to ask
whether some sections wobble significantly more than others.

In [ ]:
#@title Reliability coefficients + drift tests
def reliability_block(mat, label):
    mat = np.asarray(mat, float)
    complete = mat[~np.isnan(mat).any(axis=1)]
    ic = icc21(mat)
    out = dict(scope=label, n_indicators=mat.shape[0], n_runs=mat.shape[1],
               n_complete=len(complete),
               kripp_alpha_ordinal=krippendorff_alpha(mat, "ordinal"),
               kripp_alpha_nominal=krippendorff_alpha(mat, "nominal"),
               fleiss_kappa=fleiss_kappa(mat),
               icc21=ic.get("icc"), pairwise_exact_agreement=pairwise_agreement(mat),
               p_units=ic.get("p_units"), p_raters_drift=ic.get("p_raters"))
    if len(complete) >= 3 and complete.shape[1] >= 3:
        fr = stats.friedmanchisquare(*complete.T)
        out.update(friedman_chi2=fr.statistic, friedman_p=fr.pvalue)
    else:
        out.update(friedman_chi2=np.nan, friedman_p=np.nan)
    return out

rel_rows = [reliability_block(MATRIX, "OVERALL (all sections)")]
for s in CFG.SECTIONS:
    m = wide.loc[s].filter(like="run").to_numpy(float)
    rel_rows.append(reliability_block(m, f"Section {s} — {FRAMEWORK[s]['title']}"))
reliability = pd.DataFrame(rel_rows)
reliability.to_csv(f"{CFG.OUT_DIR}/reliability.csv", index=False)

display(reliability.set_index("scope").style.format({
    "kripp_alpha_ordinal": "{:.3f}", "kripp_alpha_nominal": "{:.3f}", "fleiss_kappa": "{:.3f}",
    "icc21": "{:.3f}", "pairwise_exact_agreement": "{:.3f}", "friedman_chi2": "{:.1f}",
    "friedman_p": "{:.4f}", "p_units": "{:.2e}", "p_raters_drift": "{:.4f}"}, na_rep="—"))

a = reliability.loc[0, "kripp_alpha_ordinal"]
verdict = ("DEPENDABLE for reporting" if a >= 0.80 else
           "TENTATIVE - use for direction only, not for individual indicator claims" if a >= 0.67 else
           "NOT USABLE for indicator-level decisions at these settings")
print(f"\nOverall Krippendorff alpha (ordinal) = {a:.3f}  ->  {verdict}")
pr = reliability.loc[0, "p_raters_drift"]; fp = reliability.loc[0, "friedman_p"]
print(f"Run-to-run drift: ICC F-test p={pr:.4f} | Friedman p={fp:.4f}  ->  "
      f"{'SIGNIFICANT systematic drift between runs' if (pd.notna(fp) and fp < CFG.ALPHA) else 'no significant systematic drift (noise is unbiased)'}")

In [ ]:
#@title Section-level wobble
# per-run section mean (a coaching report quotes these, so their wobble is what matters)
# reindexed over every iteration so an all-NA iteration stays a column instead of
# silently shortening the series (charts index these positionally against MATRIX)
_ok = scores_long.dropna(subset=["score"])
sec_run = (_ok.groupby(["section", "iteration"], observed=True)["score"].mean().unstack()
           .reindex(index=list(CFG.SECTIONS), columns=range(CFG.N_ITERATIONS)))
overall_run = _ok.groupby("iteration")["score"].mean().reindex(range(CFG.N_ITERATIONS))

sec_rows = []
for s in CFG.SECTIONS:
    xs = sec_run.loc[s].to_numpy(float)
    lo, hi = boot_ci(xs)
    sub = ind_stats[ind_stats.section == s]
    sec_rows.append(dict(
        section=s, title=FRAMEWORK[s]["title"], n_indicators=len(sub),
        mean=float(np.nanmean(xs)), sd_across_runs=float(np.nanstd(xs, ddof=1)),
        cv=float(np.nanstd(xs, ddof=1) / np.nanmean(xs)),
        ci_lo=lo, ci_hi=hi, ci_width=hi - lo,
        min_run=float(np.nanmin(xs)), max_run=float(np.nanmax(xs)),
        mean_indicator_sd=float(sub.sd.mean()),
        pct_unstable=float((sub.grade != "stable").mean()),
        pct_sig_wobble=float(sub.sig_wobble.mean()),
        mean_flip_rate=float(sub.flip_rate.mean()),
        na_rate=float(sub.na_rate.mean()),
        kripp_alpha=reliability.loc[reliability.scope.str.startswith(f"Section {s}"),
                                    "kripp_alpha_ordinal"].iloc[0]))
xs = overall_run.dropna().to_numpy(float); lo, hi = boot_ci(xs)
sec_rows.append(dict(section="ALL", title="All four sections", n_indicators=len(ind_stats),
                     mean=float(xs.mean()), sd_across_runs=float(xs.std(ddof=1)),
                     cv=float(xs.std(ddof=1) / xs.mean()), ci_lo=lo, ci_hi=hi, ci_width=hi - lo,
                     min_run=float(xs.min()), max_run=float(xs.max()),
                     mean_indicator_sd=float(ind_stats.sd.mean()),
                     pct_unstable=float((ind_stats.grade != "stable").mean()),
                     pct_sig_wobble=float(ind_stats.sig_wobble.mean()),
                     mean_flip_rate=float(ind_stats.flip_rate.mean()),
                     na_rate=float(ind_stats.na_rate.mean()),
                     kripp_alpha=reliability.loc[0, "kripp_alpha_ordinal"]))

sec_stats = pd.DataFrame(sec_rows)
sec_stats.to_csv(f"{CFG.OUT_DIR}/section_wobble.csv", index=False)
display(sec_stats.set_index("section").style.format({
    c: "{:.3f}" for c in ["mean", "sd_across_runs", "cv", "ci_lo", "ci_hi", "ci_width",
                          "min_run", "max_run", "mean_indicator_sd", "pct_unstable",
                          "pct_sig_wobble", "mean_flip_rate", "na_rate", "kripp_alpha"]}))

# Is one section significantly wobblier than another? Levene on within-indicator deviations.
dev = (scores_long.dropna(subset=["score"]).copy())
dev["abs_dev"] = (dev.score - dev.groupby("code", observed=True).score.transform("mean")).abs()
groups = [g.abs_dev.to_numpy() for _, g in dev.groupby("section", observed=True) if len(g) > 1]
if len(groups) >= 2:
    lev = stats.levene(*groups, center="median")
    kw  = stats.kruskal(*groups)
    print(f"\nLevene (equal wobble across sections): W={lev.statistic:.2f}, p={lev.pvalue:.4f}")
    print(f"Kruskal-Wallis on |deviation|:          H={kw.statistic:.2f}, p={kw.pvalue:.4f}")
    print("=> " + ("sections differ significantly in how much they wobble - "
                   "look at mean_indicator_sd above to see which"
                   if lev.pvalue < CFG.ALPHA else
                   "no significant difference in wobble between sections"))

In [ ]:
#@title Headline numbers
n_sig   = int(ind_stats.sig_wobble.sum())
n_tot   = int(ind_stats.p_wobble.notna().sum())
flippy  = ind_stats[ind_stats.flip_rate > 0].sort_values("flip_rate", ascending=False)
worst   = ind_stats.nlargest(5, "sd")[["code", "indicator", "mean", "sd", "ci_lo", "ci_hi", "grade"]]

HEADLINE = {
    "model": REPO, "quant": MODEL_SPEC["quant"], "temperature": CFG.TEMPERATURE,
    "iterations": CFG.N_ITERATIONS, "indicators": len(ind_stats),
    "overall_mean": round(float(overall_run.mean()), 3),
    "overall_ci": [round(v, 3) for v in boot_ci(overall_run.dropna().to_numpy(float))],
    "overall_sd_across_runs": round(float(overall_run.std(ddof=1)), 3),
    "mean_indicator_sd": round(float(ind_stats.sd.mean()), 3),
    "kripp_alpha_ordinal": round(float(reliability.loc[0, "kripp_alpha_ordinal"]), 3),
    "icc21": round(float(reliability.loc[0, "icc21"]), 3),
    "pairwise_exact_agreement": round(float(reliability.loc[0, "pairwise_exact_agreement"]), 3),
    "pct_indicators_fully_stable": round(float((ind_stats.grade == "stable").mean()), 3),
    "pct_indicators_sig_wobble": round(n_sig / max(n_tot, 1), 3),
    "n_indicators_flipping_proficiency": int((ind_stats.flip_rate > 0).sum()),
    "mean_ci_width": round(float(ind_stats.ci_width.mean()), 3),
    "na_rate": round(float(ind_stats.na_rate.mean()), 3),
    "parse_failure_rate": round(float((scores_long.parse == "none").mean()), 4),
}
print(json.dumps(HEADLINE, indent=2))
with open(f"{CFG.OUT_DIR}/headline.json", "w") as f: json.dump(HEADLINE, f, indent=2)

print(f"\nSignificant wobble (Holm q<{CFG.ALPHA}): {n_sig}/{n_tot} indicators")
print("\nWobbliest five:"); display(worst)
print(f"\nIndicators whose Proficient (>={CFG.PROFICIENCY_CUT}) verdict flips between runs "
      f"— these are the ones you cannot report from a single pass:")
display(flippy[["code", "indicator", "mode", "mean", "p_proficient", "flip_rate", "sd"]])

---
## 10 · Charts

One theme, applied to every figure. Colours are a validated palette: four categorical hues for
the four sections (blue / orange / aqua / violet — checked for colour-vision-deficiency
separation on all pairs), a single-hue ordinal blue ramp for the 1→4 scale (ordered data gets an
ordered ramp, never a rainbow), and the reserved status palette only where a colour genuinely
means good/bad. Every chart has a table twin in §8–§9, so no value is reachable by colour alone.
All figures are also written to `wobble_out/*.png` at 200 dpi.

In [ ]:
#@title Theme
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

SURFACE, PLANE = "#fcfcfb", "#f9f9f7"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"

SERIES  = {"B": "#2a78d6", "C": "#eb6834", "D": "#1baf7a", "F": "#4a3aa7"}   # validated all-pairs
ORDINAL = ["#86b6ef", "#3987e5", "#256abf", "#0d366b"]                       # scores 1,2,3,4
STATUS  = {"stable": "#0ca30c", "minor": "#fab219", "material": "#ec835a", "critical": "#d03b3b"}
STATUS["severe"] = STATUS["critical"]
GRADE_MARK = {"stable": "●", "minor": "▲", "material": "◆", "severe": "✕"}
LEVEL_NAME = ["Not observed", "Developing", "Proficient", "Highly effective"]
SECTION_SHORT = {"B": "Lesson Plan Fidelity", "C": "High-Leverage Practices",
                 "D": "Student Engagement", "F": "Teacher Subject Knowledge"}

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 9.5,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.axisbelow": True, "xtick.major.size": 0, "ytick.major.size": 0,
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200,
    "savefig.bbox": "tight", "axes.titlelocation": "left", "axes.titlepad": 14,
    "axes.titlesize": 12.5, "axes.titleweight": "semibold",
})

def style(ax, xgrid=False, ygrid=False, spines=("left", "bottom")):
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(s in spines)
    ax.xaxis.grid(xgrid); ax.yaxis.grid(ygrid)
    return ax

# Offsets are computed in POINTS, not axes fractions: these figures range from 4 to 14 inches
# tall, and an axes-fraction offset that looks right on one silently collapses on the other.
def _axh(fig, ax):
    return max(ax.get_position().height * fig.get_figheight(), 0.1)

def titles(ax, title, sub=None):
    ax.set_title(title, pad=30)
    ax.annotate(SUBTITLE if sub is None else sub, xy=(0, 1), xycoords="axes fraction",
                xytext=(0, 11), textcoords="offset points", color=MUTED, fontsize=8.5,
                va="bottom", ha="left", annotation_clip=False)

def legend_below(fig, ax, handles=None, pts=40, ncol=4, **kw):
    off = (pts / 72.0) / _axh(fig, ax)
    kw.setdefault("fontsize", 8.5)
    args = dict(loc="upper left", bbox_to_anchor=(0, -off), ncol=ncol, **kw)
    return ax.legend(handles=handles, **args) if handles is not None else ax.legend(**args)

def caption(fig, ax, text, pts=72):
    pos = ax.get_position()
    fig.text(pos.x0, pos.y0 - (pts / 72.0) / fig.get_figheight(),
             textwrap.fill(text, int(fig.get_figwidth() * 13.5)),
             color=MUTED, fontsize=8.5, ha="left", va="top", linespacing=1.5)

def save(fig, name):
    fig.savefig(f"{CFG.OUT_DIR}/{name}.png")
    return fig

SUBTITLE = (f"{REPO.split('/')[-1]} · {MODEL_SPEC['quant']} · T={CFG.TEMPERATURE} · "
            f"top_p={CFG.TOP_P} · {CFG.N_ITERATIONS} runs · {CFG.SCORING_MODE} · "
            f"session {SESSION_META['session_id'][:8]}")
print(SUBTITLE)

In [ ]:
#@title Chart 1 — headline stat tiles (the numbers are the chart)
tiles = [
    ("Overall mean score",     f"{HEADLINE['overall_mean']:.2f}",
     f"95% CI {HEADLINE['overall_ci'][0]:.2f}–{HEADLINE['overall_ci'][1]:.2f}  ·  out of 4.00", None),
    ("Krippendorff α (ordinal)", f"{HEADLINE['kripp_alpha_ordinal']:.2f}",
     "≥.80 dependable · .67–.80 tentative",
     "stable" if HEADLINE['kripp_alpha_ordinal'] >= .80 else
     "minor"  if HEADLINE['kripp_alpha_ordinal'] >= .67 else "critical"),
    ("Mean indicator SD",      f"{HEADLINE['mean_indicator_sd']:.2f}",
     "rubric points of run-to-run noise",
     "stable" if HEADLINE['mean_indicator_sd'] < .25 else
     "minor"  if HEADLINE['mean_indicator_sd'] < .50 else
     "material" if HEADLINE['mean_indicator_sd'] < .80 else "critical"),
    ("Fully stable indicators", f"{HEADLINE['pct_indicators_fully_stable']*100:.0f}%",
     f"identical in all {CFG.N_ITERATIONS} runs",
     "stable" if HEADLINE['pct_indicators_fully_stable'] >= .8 else
     "minor"  if HEADLINE['pct_indicators_fully_stable'] >= .5 else "material"),
    ("Significant wobble",     f"{HEADLINE['pct_indicators_sig_wobble']*100:.0f}%",
     f"Holm-corrected q<{CFG.ALPHA} vs a ≤5% noise floor",
     "stable" if HEADLINE['pct_indicators_sig_wobble'] <= .1 else
     "material" if HEADLINE['pct_indicators_sig_wobble'] <= .4 else "critical"),
    ("Proficiency verdict flips", f"{HEADLINE['n_indicators_flipping_proficiency']}",
     f"of {len(ind_stats)} indicators cross the ≥3 line",
     "stable" if HEADLINE['n_indicators_flipping_proficiency'] == 0 else
     "minor"  if HEADLINE['n_indicators_flipping_proficiency'] <= 3 else "critical"),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 4.6))
fig.subplots_adjust(wspace=0.30, hspace=0.55)
for ax, (label, value, sub, st) in zip(axes.ravel(), tiles):
    ax.set_axis_off(); ax.set_facecolor(SURFACE)
    ax.text(0.055, 1.00, label.upper(), color=MUTED, fontsize=8.2, va="top", transform=ax.transAxes)
    ax.text(0.055, 0.80, value, color=INK, fontsize=27, va="top", ha="left",
            transform=ax.transAxes, fontweight="medium")
    ax.text(0.055, 0.30, textwrap.fill(sub, 34), color=INK2, fontsize=8.0, va="top",
            linespacing=1.5, transform=ax.transAxes)
    if st:                                    # status = icon + colour + label, never colour alone
        ax.text(0.055, -0.02, f"{GRADE_MARK.get(st, '●')} {st}", color=STATUS[st], fontsize=9,
                va="top", transform=ax.transAxes, fontweight="semibold")
    ax.plot([0, 0], [-0.06, 1.04], transform=ax.transAxes, color=GRID, lw=1.2, clip_on=False)
fig.suptitle("Scoring wobble — headline", x=0.008, ha="left", y=1.10, fontsize=13.5,
             fontweight="semibold")
fig.text(0.008, 1.015, SUBTITLE, color=MUTED, fontsize=8.5, ha="left")
save(fig, "01_headline"); plt.show()

In [ ]:
#@title Chart 2 — score matrix: every indicator x every run
n_runs = MATRIX.shape[1]
fig, ax = plt.subplots(figsize=(max(7.0, 2.0 + 0.55 * n_runs), 0.30 * len(IND_CODES) + 1.8))

cmap = ListedColormap(ORDINAL); cmap.set_bad("#eceae4")
norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)
ax.pcolormesh(np.arange(n_runs + 1), np.arange(len(IND_CODES) + 1),
              np.ma.masked_invalid(MATRIX)[::-1], cmap=cmap, norm=norm,
              edgecolors=SURFACE, linewidth=2.0)

if len(IND_CODES) * n_runs <= 480:                       # label only when the cells fit
    for i, row in enumerate(MATRIX[::-1]):
        for j, v in enumerate(row):
            if np.isnan(v):
                ax.text(j + .5, i + .5, "NA", ha="center", va="center", color=MUTED, fontsize=7)
            else:
                ax.text(j + .5, i + .5, f"{int(v)}", ha="center", va="center",
                        color="#ffffff" if v >= 3 else INK, fontsize=7.5)

ax.set_yticks(np.arange(len(IND_CODES)) + .5)
ax.set_yticklabels([f"{c}  {CODE2NAME[c][:42]}" for c in IND_CODES][::-1], fontsize=8)
for t, c in zip(ax.get_yticklabels(), IND_CODES[::-1]):
    t.set_color(SERIES[CODE2SECTION[c]])
ax.set_xticks(np.arange(n_runs) + .5)
ax.set_xticklabels([f"r{i+1}" for i in range(n_runs)], fontsize=8)
ax.set_xlim(0, n_runs); ax.set_ylim(0, len(IND_CODES))
style(ax, spines=())
titles(ax, "Score matrix — where the wobble actually is")
legend_below(fig, ax, pts=34, ncol=5,
             handles=[Patch(facecolor=ORDINAL[i], label=f"{i+1} — {LEVEL_NAME[i]}")
                      for i in range(4)] + [Patch(facecolor="#eceae4", edgecolor=BASELINE, lw=.8,
                                              label="NA / unparsed")])
caption(fig, ax, "A row of one colour is a stable indicator. Any change of shade along a row is "
                 "pure sampling noise: the transcript, rubric and prompt were identical in every "
                 "run. Row labels are coloured by section (B blue · C orange · D aqua · F violet).",
        pts=64)
save(fig, "02_score_matrix"); plt.show()

In [ ]:
#@title Chart 3 — mean score with bootstrap 95% CI (the interval is the point)
d = ind_stats.dropna(subset=["mean"]).copy()
d["ypos"] = np.arange(len(d))[::-1]

fig, ax = plt.subplots(figsize=(9.5, 0.30 * len(d) + 1.9))
ax.axvspan(CFG.PROFICIENCY_CUT, 4.35, color="#f4f3ef", zorder=0)
ax.axvline(CFG.PROFICIENCY_CUT, color=BASELINE, lw=1.4, ls=(0, (5, 3)), zorder=1)
ax.text(CFG.PROFICIENCY_CUT + .05, len(d) - .4, "Proficient ≥ 3", color=MUTED, fontsize=8.5,
        va="top")

for r in d.itertuples():
    col = SERIES[r.section]
    ax.plot([r.min, r.max], [r.ypos, r.ypos], color=col, lw=0.9, alpha=.30, zorder=2)
    ax.plot([r.ci_lo, r.ci_hi], [r.ypos, r.ypos], color=col, lw=2.4, solid_capstyle="round",
            alpha=.55, zorder=3)
    ax.plot(r.mean, r.ypos, "o", ms=7.5, color=col, mec=SURFACE, mew=2.0, zorder=4)

ax.set_yticks(d.ypos)
ax.set_yticklabels([f"{r.code}  {r.indicator[:40]}" for r in d.itertuples()], fontsize=8)
for t, s in zip(ax.get_yticklabels(), d.section):
    t.set_color(SERIES[s])
ax.set_ylim(-.8, len(d) - .2)
ax.set_xlim(0.7, 4.35); ax.set_xticks([1, 2, 3, 4])
ax.set_xlabel("score  (1 = not observed  →  4 = highly effective)")
style(ax, xgrid=True, spines=("bottom",))
titles(ax, "Per-indicator mean and 95% bootstrap CI")
legend_below(fig, ax, pts=52, ncol=3,
             handles=[Line2D([], [], marker="o", ls="none", ms=7, color=SERIES[s], mec=SURFACE,
                             mew=1.6, label=f"Section {s} — {SECTION_SHORT[s]}")
                      for s in CFG.SECTIONS]
             + [Line2D([], [], color=MUTED, lw=2.4, alpha=.55, label="95% CI of the mean"),
                Line2D([], [], color=MUTED, lw=0.9, alpha=.4, label="observed min–max")])
caption(fig, ax, "Thick bar = bootstrap 95% CI of the mean; hairline = full observed range; "
                 "dot = mean. Any CI straddling the dashed line is an indicator whose Proficient "
                 "verdict is not resolvable at these settings — raise N_ITERATIONS to narrow it, "
                 "or lower TEMPERATURE to shrink the underlying wobble.", pts=102)
save(fig, "03_mean_ci"); plt.show()

In [ ]:
#@title Chart 4 — wobble magnitude per indicator, graded
d = ind_stats.sort_values("sd", ascending=True, na_position="first").copy()
d["ypos"] = np.arange(len(d))

fig, ax = plt.subplots(figsize=(9.0, 0.29 * len(d) + 1.9))
for xv, lab in ((0.25, "negligible"), (0.50, "material"), (0.80, "severe")):
    ax.axvline(xv, color=BASELINE, lw=1.0, ls=(0, (4, 3)), zorder=1)
    ax.text(xv, len(d) - .35, f" {lab}", color=MUTED, fontsize=8, va="bottom")

ax.barh(d.ypos, d.sd.fillna(0), height=0.62, color=[STATUS[g] for g in d.grade], zorder=2,
        edgecolor=SURFACE, linewidth=2.0)
for r in d.itertuples():
    txt = f"{GRADE_MARK[r.grade]} {r.sd:.2f}" if pd.notna(r.sd) else f"{GRADE_MARK[r.grade]} no data"
    ax.text((r.sd if pd.notna(r.sd) else 0) + 0.012, r.ypos, txt, va="center", fontsize=7.8,
            color=INK2)

ax.set_yticks(d.ypos)
ax.set_yticklabels([f"{r.code}  {r.indicator[:40]}" for r in d.itertuples()], fontsize=8)
ax.set_xlabel("SD of the score across runs  (rubric points)")
_sdmax = float(np.nanmax(d.sd.to_numpy(float))) if d.sd.notna().any() else 0.0
ax.set_xlim(0, max(0.95, _sdmax * 1.28)); ax.set_ylim(-.8, len(d) - .2)
style(ax, xgrid=True, spines=("bottom",))
titles(ax, "How much each indicator wobbles")
legend_below(fig, ax, pts=52, ncol=2,
             handles=[Patch(facecolor=STATUS[g], label=f"{GRADE_MARK[g]} {g}"
                            + {"stable": " — identical in every run",
                               "minor": " — ≥80% modal, spread ≤1 band",
                               "material": " — ≥60% modal, spread ≤2 bands",
                               "severe": " — below that, or mostly NA"}[g])
                      for g in ("stable", "minor", "material", "severe")])
caption(fig, ax, "Grade combines modal agreement with spread, so it separates “always 3” from "
                 "“3 six times out of ten”. Each bar carries its own icon and value, so the grade "
                 "never depends on colour alone. SD around 0.5 means a report built on a single "
                 "run is roughly a coin-flip away from a different band on that indicator.",
        pts=102)
save(fig, "04_wobble_sd"); plt.show()

In [ ]:
#@title Chart 5 — score distribution per indicator (100% stacked)
d = ind_stats.copy()
props = np.zeros((len(d), 5))                       # columns: 1, 2, 3, 4, NA
for i, code_ in enumerate(d.code):
    xs = MATRIX[IND_CODES.index(code_)]
    valid = np.isfinite(xs)
    if valid.any():
        for j, lv in enumerate(LEVELS):
            props[i, j] = (xs[valid] == lv).mean() * valid.mean()
    props[i, 4] = 1.0 - props[i, :4].sum()

ypos = np.arange(len(d))[::-1]
fig, ax = plt.subplots(figsize=(8.8, 0.29 * len(d) + 1.9))
left = np.zeros(len(d))
for j, (col, lab) in enumerate(zip(ORDINAL + ["#eceae4"],
                                   [f"{i+1} {LEVEL_NAME[i]}" for i in range(4)] + ["NA"])):
    ax.barh(ypos, props[:, j], left=left, height=0.66, color=col, label=lab,
            edgecolor=SURFACE, linewidth=2.0, zorder=2)
    left += props[:, j]

for i, r in enumerate(d.itertuples()):              # label the modal share only, never every cell
    ax.text(1.015, ypos[i], f"{r.modal_share*100:.0f}% @ {int(r.mode)}" if r.n else "no data",
            va="center", fontsize=7.8, color=INK2)

ax.set_yticks(ypos)
ax.set_yticklabels([f"{r.code}  {r.indicator[:38]}" for r in d.itertuples()], fontsize=8)
for t, s in zip(ax.get_yticklabels(), d.section):
    t.set_color(SERIES[s])
ax.set_ylim(-.8, len(d) - .2)
ax.set_xlim(0, 1.0); ax.set_xticks([0, .25, .5, .75, 1])
ax.set_xticklabels(["0", "25%", "50%", "75%", "100%"])
ax.set_xlabel("share of runs")
style(ax, xgrid=True, spines=("bottom",))
titles(ax, "Which levels the model actually chose")
legend_below(fig, ax, pts=52, ncol=5)
caption(fig, ax, "A single full-width block is a decided indicator. Two adjacent shades is "
                 "ordinary boundary uncertainty. Non-adjacent shades (1 and 3, or 2 and 4) mean "
                 "the model is not reading the same evidence twice — check its evidence strings in "
                 "§8 before trusting the mean.", pts=102)
save(fig, "05_distribution"); plt.show()

In [ ]:
#@title Chart 6 — section scores: every run, plus mean and CI
fig, ax = plt.subplots(figsize=(8.8, 4.6))
jit = np.random.default_rng(CFG.STATS_SEED).uniform(-.13, .13, size=(len(CFG.SECTIONS), n_runs))

for i, s in enumerate(CFG.SECTIONS):
    xs, col = sec_run.loc[s].to_numpy(float), SERIES[s]
    row = sec_stats[sec_stats.section == s].iloc[0]
    ax.plot(np.full(n_runs, i) + jit[i], xs, "o", ms=6, color=col, alpha=.42, mec=SURFACE,
            mew=1.6, zorder=2)
    ax.plot([i, i], [row.ci_lo, row.ci_hi], color=col, lw=9, alpha=.20, solid_capstyle="round",
            zorder=3)
    ax.plot([i - .30, i + .30], [row["mean"]] * 2, color=col, lw=2.6, solid_capstyle="round",
            zorder=4)
    ax.text(i + .36, row["mean"], f"{row['mean']:.2f}\n±{row.ci_width/2:.2f}", fontsize=8.5,
            color=col, va="center", fontweight="semibold")

allrow = sec_stats[sec_stats.section == "ALL"].iloc[0]
ax.axhline(allrow["mean"], color=BASELINE, lw=1.0, ls=(0, (4, 3)), zorder=1)
ax.text(len(CFG.SECTIONS) - .55, allrow["mean"], f"all-section mean {allrow['mean']:.2f} ",
        color=MUTED, fontsize=8, va="bottom", ha="right")

ax.set_xticks(range(len(CFG.SECTIONS)))
ax.set_xticklabels([f"{s}\n{FRAMEWORK[s]['title'][:22]}" for s in CFG.SECTIONS], fontsize=8.5,
                   color=INK2)
ax.set_xlim(-.55, len(CFG.SECTIONS) - .25)
ax.set_ylabel("section mean score"); ax.set_ylim(1, 4); ax.set_yticks([1, 2, 3, 4])
style(ax, ygrid=True, spines=("left",))
titles(ax, "Section means — one dot per run")
caption(fig, ax, "Faint dots are the individual runs, the bar is the mean, the band is its "
                 "bootstrap 95% CI. Averaging 7–12 indicators cancels a lot of noise, which is why "
                 "section means are far tighter than the indicators inside them — report at this "
                 "level when the indicator CIs are too wide to defend.", pts=54)
save(fig, "06_section_runs"); plt.show()

In [ ]:
#@title Chart 7 — drift: does a later run score systematically differently?
fig, ax = plt.subplots(figsize=(9.2, 4.6))
x = np.arange(1, n_runs + 1)
ax.plot(x, overall_run.to_numpy(float), "-", lw=1.4, color=MUTED, alpha=.85, label="all sections",
        zorder=2)
for s in CFG.SECTIONS:
    ys = sec_run.loc[s].to_numpy(float)
    ax.plot(x, ys, "-o", lw=2.0, ms=6, color=SERIES[s], mec=SURFACE, mew=1.8,
            label=f"Section {s}", zorder=3)
    ax.text(x[-1] + .14, ys[-1], s, color=SERIES[s], fontsize=9, va="center",
            fontweight="semibold")

ax.set_xticks(x); ax.set_xlabel("iteration"); ax.set_ylabel("mean score")
ax.set_xlim(.7, n_runs + .6); ax.set_ylim(1, 4); ax.set_yticks([1, 2, 3, 4])
style(ax, ygrid=True, spines=("left", "bottom"))
titles(ax, "Run-to-run drift")
legend_below(fig, ax, pts=48, ncol=5)
fr_p, ic_p = reliability.loc[0, "friedman_p"], reliability.loc[0, "p_raters_drift"]
caption(fig, ax, f"Friedman p={fr_p:.4f} · ICC rater F-test p={ic_p:.4f}. "
        + ("Runs differ systematically — the noise has a direction, so a single run is biased, "
           "not merely imprecise, and averaging converges on that bias."
           if pd.notna(fr_p) and fr_p < CFG.ALPHA else
           "No significant systematic difference between runs: the noise is unbiased, so "
           "averaging across runs converges on the right answer at roughly √N."), pts=76)
save(fig, "07_drift"); plt.show()

---
## 11 · Hyperparameter sweep (optional)

The single most useful follow-up: **how does wobble respond to temperature?** This cell
re-runs the whole framework at each temperature in `SWEEP_TEMPS` with `SWEEP_ITERS`
iterations each, and plots wobble and reliability against it.

Cost: `len(SWEEP_TEMPS) × SWEEP_ITERS` iterations. The defaults (4 temps × 5 iterations
= 20 iterations ≈ 35–50 min on a T4) are sized for one Colab session. Set `RUN_SWEEP = False`
to skip.

Read it as a calibration curve. `T=0` is the control arm: wobble there is the floor imposed by
kernel non-determinism, not by sampling. What you are looking for is the largest temperature
whose α still clears your reporting bar — and whether *any* temperature does.

The same pattern works for other knobs: swap `SWEEP_TEMPS` for a list of `TOP_P` values,
`PROMPT_VARIANT`s (`"standard"`/`"terse"`/`"cot"`), `INDICATOR_ORDER`s, or `MODEL_KEY`s —
change the two marked lines in `sweep_one()`.

In [ ]:
#@title Sweep
RUN_SWEEP   = False           #@param {type:"boolean"}
SWEEP_TEMPS = [0.0, 0.3, 0.7, 1.0]
SWEEP_ITERS = 5

def summarise(df):
    """Compact wobble summary from a long score frame."""
    w = (df.pivot_table(index="code", columns="iteration", values="score", observed=True)
         .reindex([c for c in ALL_CODES if c in set(df.code)]))
    M = w.to_numpy(float)
    sds, shares, flips = [], [], []
    for xs in M:
        x = xs[~np.isnan(xs)]
        if len(x) < 2:
            continue
        sds.append(x.std(ddof=1))
        shares.append((x == stats.mode(x, keepdims=False).mode).mean())
        p = (x >= CFG.PROFICIENCY_CUT).mean(); flips.append(2 * min(p, 1 - p))
    runmeans = df.dropna(subset=["score"]).groupby("iteration").score.mean()
    return dict(mean_indicator_sd=float(np.mean(sds)), modal_share=float(np.mean(shares)),
                mean_flip_rate=float(np.mean(flips)),
                kripp_alpha=krippendorff_alpha(M, "ordinal"),
                icc21=icc21(M).get("icc"), exact_agreement=pairwise_agreement(M),
                overall_mean=float(runmeans.mean()),
                overall_sd=float(runmeans.std(ddof=1)) if len(runmeans) > 1 else 0.0,
                na_rate=float(df.score.isna().mean()),
                pct_stable=float(np.mean([s == 1.0 for s in shares])))

def sweep_one(value, n_iter, seed_offset):
    saved = (CFG.TEMPERATURE, CFG.N_ITERATIONS, CFG.BASE_SEED)
    CFG.TEMPERATURE = value            # <-- change this line to sweep a different knob
    CFG.N_ITERATIONS = n_iter
    CFG.BASE_SEED = saved[2] + seed_offset
    try:
        recs = []
        for it in range(n_iter):
            rows, _ = run_iteration(it)
            recs += rows
        df = pd.DataFrame(recs)
        df["score"] = pd.to_numeric(df["score"], errors="coerce")
        return df
    finally:
        CFG.TEMPERATURE, CFG.N_ITERATIONS, CFG.BASE_SEED = saved

sweep = None
if RUN_SWEEP:
    out, t0 = [], time.time()
    for k, temp in enumerate(SWEEP_TEMPS):
        print(f"\n=== sweep T={temp} ({k+1}/{len(SWEEP_TEMPS)}) ===")
        df = sweep_one(temp, SWEEP_ITERS, seed_offset=50_000 * (k + 1))
        df.to_csv(f"{CFG.OUT_DIR}/sweep_T{temp}.csv", index=False)
        out.append(dict(knob="temperature", value=temp, n_iter=SWEEP_ITERS, **summarise(df)))
        print("   ", {k2: (round(v, 3) if isinstance(v, float) else v)
                      for k2, v in out[-1].items() if k2 not in ("knob", "n_iter")})
    sweep = pd.DataFrame(out)
    sweep.to_csv(f"{CFG.OUT_DIR}/sweep_summary.csv", index=False)
    print(f"\nsweep done in {(time.time()-t0)/60:.1f} min")
    display(sweep.style.format({c: "{:.3f}" for c in sweep.columns if c != "knob"}))
else:
    print("RUN_SWEEP is False — skipped. Flip it to True to get the calibration curve.")

In [ ]:
#@title Chart 8 — wobble vs temperature (only if the sweep ran)
if sweep is not None and len(sweep) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.1))
    panels = [
        ("mean_indicator_sd", "Mean indicator SD", "rubric points  (higher = wobblier)",
         SERIES["C"], None),
        ("kripp_alpha", "Krippendorff α (ordinal)", "α  (higher = more reliable)",
         SERIES["B"], 0.80),
        ("mean_flip_rate", "Proficiency flip rate", "share  (higher = less decidable)",
         SERIES["F"], None),
    ]
    for ax, (col, title, ylab, colr, ref) in zip(axes, panels):
        ax.plot(sweep.value, sweep[col], "-o", lw=2.0, ms=7, color=colr, mec=SURFACE, mew=2.0)
        for xv, yv in zip(sweep.value, sweep[col]):
            ax.annotate(f"{yv:.2f}", (xv, yv), textcoords="offset points", xytext=(0, 9),
                        ha="center", fontsize=8, color=INK2)
        if ref is not None:
            ax.axhline(ref, color=BASELINE, lw=1.0, ls=(0, (4, 3)))
            ax.text(sweep.value.max(), ref, " dependable ≥.80", color=MUTED, fontsize=8,
                    va="bottom", ha="right")
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("temperature"); ax.set_ylabel(ylab)
        ax.set_xticks(list(sweep.value))
        ax.margins(x=0.10, y=0.20)          # room for the point labels
        style(ax, ygrid=True, spines=("left", "bottom"))
    fig.suptitle("Wobble as a function of temperature", x=0.005, ha="left", y=1.06,
                 fontsize=13.5, fontweight="semibold")
    fig.text(0.005, 0.99, f"{REPO.split('/')[-1]} · {SWEEP_ITERS} runs per temperature · "
                          f"{len(ind_stats)} indicators", color=MUTED, fontsize=8.5, ha="left")
    fig.tight_layout()
    save(fig, "08_temperature_sweep"); plt.show()
else:
    print("No sweep results to plot.")

---
## 12 · Export

In [ ]:
#@title Zip everything and download
import shutil, glob
print("\n".join(sorted(os.path.basename(p) for p in glob.glob(f"{CFG.OUT_DIR}/*"))))
archive = shutil.make_archive(f"wobble_{SESSION_META['session_id'][:8]}_T{CFG.TEMPERATURE}",
                              "zip", CFG.OUT_DIR)
print("\n->", archive, f"({os.path.getsize(archive)/1e6:.2f} MB)")
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("(not on Colab, or download blocked — grab the file from the Files pane)", e)

---
## 13 · How to read the output

**The one number to look at first** is overall **Krippendorff's α (ordinal)** in §9.2.

| α | What it licenses |
|---|---|
| **≥ 0.80** | Report individual indicator scores from a single pass. |
| **0.67 – 0.80** | Report *section* means and mode-of-N indicator scores. Do not put a single-pass indicator score in front of a teacher. |
| **< 0.67** | Indicator-level output is not decision-grade at these settings. Use section means, raise N and take the mode, lower the temperature, or move to a stronger model. |

**The distinctions that matter when you write this up**

- *Wobble ≠ inaccuracy.* Everything here measures **precision** (agreement with itself). A model
  can be perfectly stable and perfectly wrong. To measure accuracy you need human-scored
  sessions and a second analysis — Cohen's quadratic-weighted κ against the human score,
  same 1–4 scale. Nothing in this notebook substitutes for that.
- *SD vs flip rate.* SD is the statistician's number; **flip rate** is the coaching one. An
  indicator with SD 0.5 that only ever moves between 3 and 4 never changes the verdict. One that
  moves between 2 and 3 changes it every other run. Chart 4 gives the first, §9.1's `flip_rate`
  the second.
- *Unbiased vs drifting noise.* If the Friedman / ICC rater tests come back non-significant, the
  noise is unbiased and **averaging converges** — so N passes buy you accuracy at √N. If they are
  significant, individual passes are biased, and averaging converges on the wrong thing.
- *Wide CI ≠ big wobble.* A CI narrows with √N. If `ci_width` is wide but SD is small, run more
  iterations. If SD is large, more iterations only measure the wobble more precisely — they don't
  reduce it.
- *NA rate is a result, not an error.* F5/F6 (Math/Science) genuinely do not apply to this English
  reading lesson. An NA rate near 1.0 there is the model being right. An NA rate near 0.5 anywhere
  means it cannot decide whether an indicator applies — that is a rubric-clarity problem, not a
  sampling one.
- *The transcript is a ceiling.* Sections C12 (space/seating), D1/D7 (visible engagement, gender)
  and B8/C10 (physical resources, tech) depend on things audio does not carry. Persistently low
  scores there are a **method limit, not a teacher finding** — expect a floor effect, and read
  those indicators as "not evidenced in audio" rather than "not observed in the classroom".

**Suggested next experiments**

1. `RUN_SWEEP = True` — get the calibration curve, pick the temperature.
2. `SCORING_MODE = "per_indicator"` — removes cross-indicator contamination. If wobble drops a
   lot, the per-section prompt was letting indicators anchor on each other.
3. `PROMPT_VARIANT = "cot"` — usually tightens agreement at a real token cost.
4. `INDICATOR_ORDER = "shuffled"` — if wobble jumps, you have order sensitivity on top of
   sampling noise, and the rubric text is doing less work than the position in the list.
5. `MODEL_KEY = "gemma2-2b"` vs `"llama3.1-8b"` vs `"qwen2.5-7b"` at fixed temperature — the
   model-size arm. Keep `DIGEST_ONCE = True` so the input is identical across arms.
6. Run 5–10 different sessions and check whether the *same* indicators wobble every time. Ones
   that do have a rubric problem (their level descriptors don't discriminate); ones that wobble
   only on some sessions have an evidence problem (that lesson didn't show enough).